In [ ]:
# Waveform and Firing Rate Clustering Analysis
# This notebook performs comprehensive clustering analysis combining waveform data with firing rate and bursting metrics

import pynapple as nap
from spikeinterface import load_sorting_analyzer
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import fnmatch
import matplotlib as mpl
import re
import os
from scipy import stats
from sklearn.decomposition import PCA, FastICA
from scipy.linalg import norm
from scipy.stats import kstest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import umap
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully!")

# Bursting capacity metrics based on autocorrelograms
def calculate_autocorrelogram(spike_times, bin_size=0.001, window_size=0.1):
    """
    Calculate autocorrelogram for spike times.
    
    Parameters:
    - spike_times: array of spike times in seconds
    - bin_size: bin size in seconds (default 1ms)
    - window_size: window size in seconds (default 100ms)
    
    Returns:
    - bins: time bins
    - autocorr: autocorrelogram values
    """
    if len(spike_times) < 2:
        return np.array([0]), np.array([0])
    
    # Create bins
    bins = np.arange(-window_size, window_size + bin_size, bin_size)
    
    # Calculate autocorrelogram
    autocorr = np.zeros(len(bins) - 1)
    
    for i, spike_time in enumerate(spike_times):
        # Find all other spikes
        other_spikes = np.concatenate([spike_times[:i], spike_times[i+1:]])
        
        # Calculate time differences
        time_diffs = other_spikes - spike_time
        
        # Bin the differences
        hist, _ = np.histogram(time_diffs, bins=bins)
        autocorr += hist
    
    # Normalize by number of spikes
    autocorr = autocorr / len(spike_times)
    
    return bins[:-1], autocorr

def calculate_bursting_metrics(spike_times, bin_size=0.001, window_size=0.1):
    """
    Calculate multiple bursting capacity metrics from autocorrelogram.
    
    Returns a dictionary with various bursting metrics:
    1. Burst Index: ratio of short-interval spikes to long-interval spikes
    2. Refractory Period Violation: spikes in refractory period
    3. Burst Peak Height: height of the first peak after time 0
    4. Burst Peak Width: width of the first peak
    5. Autocorr Skewness: skewness of the autocorrelogram
    6. Short ISI Ratio: ratio of ISIs < 10ms to total ISIs
    """
    if len(spike_times) < 10:  # Need sufficient spikes for reliable metrics
        return {
            'burst_index': 0,
            'refractory_violation': 0,
            'burst_peak_height': 0,
            'burst_peak_width': 0,
            'autocorr_skewness': 0,
            'short_isi_ratio': 0,
            'autocorr_vector': np.zeros(200)  # Fixed size for consistency
        }
    
    # Calculate autocorrelogram
    bins, autocorr = calculate_autocorrelogram(spike_times, bin_size, window_size)
    
    # Find center bin (time = 0)
    center_idx = len(bins) // 2
    
    # 1. Burst Index: ratio of short-interval spikes (1-10ms) to long-interval spikes (50-100ms)
    short_window = (bins >= 0.001) & (bins <= 0.010)  # 1-10ms
    long_window = (bins >= 0.050) & (bins <= 0.100)   # 50-100ms
    
    short_count = np.sum(autocorr[short_window])
    long_count = np.sum(autocorr[long_window])
    burst_index = short_count / (long_count + 1e-10)  # Add small value to avoid division by zero
    
    # 2. Refractory Period Violation: spikes in 0-2ms window
    refractory_window = (bins >= 0) & (bins <= 0.002)
    refractory_violation = np.sum(autocorr[refractory_window])
    
    # 3. Burst Peak Height: height of the first peak after time 0
    # Look for peaks in the 1-20ms window
    peak_window = (bins >= 0.001) & (bins <= 0.020)
    if np.any(peak_window):
        burst_peak_height = np.max(autocorr[peak_window])
    else:
        burst_peak_height = 0
    
    # 4. Burst Peak Width: width at half height of the first peak
    if burst_peak_height > 0:
        half_height = burst_peak_height / 2
        # Fix the indexing issue - use the correct window for peak_indices
        peak_indices = np.where(autocorr[peak_window] >= half_height)[0]
        if len(peak_indices) > 0:
            burst_peak_width = (np.max(peak_indices) - np.min(peak_indices)) * bin_size
        else:
            burst_peak_width = 0
    else:
        burst_peak_width = 0
    
    # 5. Autocorr Skewness: skewness of the autocorrelogram
    autocorr_skewness = stats.skew(autocorr)
    
    # 6. Short ISI Ratio: ratio of ISIs < 10ms to total ISIs
    isis = np.diff(spike_times)
    short_isis = np.sum(isis < 0.010)
    total_isis = len(isis)
    short_isi_ratio = short_isis / (total_isis + 1e-10)
    
    # Create a standardized autocorr vector (fixed size for consistency)
    autocorr_vector = np.zeros(200)
    if len(autocorr) > 0:
        # Interpolate to fixed size
        autocorr_vector = np.interp(np.linspace(0, len(autocorr)-1, 200), 
                                  np.arange(len(autocorr)), autocorr)
    
    return {
        'burst_index': burst_index,
        'refractory_violation': refractory_violation,
        'burst_peak_height': burst_peak_height,
        'burst_peak_width': burst_peak_width,
        'autocorr_skewness': autocorr_skewness,
        'short_isi_ratio': short_isi_ratio,
        'autocorr_vector': autocorr_vector
    }

print("Bursting metrics functions defined")

def imec_key_sorter(key):
    """Extract the number after 'imec', default to a large number if not found."""
    match = re.search(r'imec(\d+)', key)
    return int(match.group(1)) if match else float('inf')

In [ ]:
#subject definitions for tumor cohort
nwb_paths = [
    Path("/data_store2/neuropixels/nwb/old/NP93_B1/NP93_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP95_B1/NP95_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP101_B3/NP101_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP105_B1/NP105_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP113_B1/NP113_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP114_B1/NP114_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP116_B2/NP116_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP122_B1/NP122_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP128_B1/NP128_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP129_B1/NP129_B1.nwb"),
    #Path("/data_store2/neuropixels/nwb/old/NP132_B2/NP132_B2.nwb"),
    #Path("/data_store2/neuropixels/nwb/old/NP132_B3/NP132_B3.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP136_B1/NP136_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP137_B1/NP137_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP138_B1/NP138_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B1/NP139_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP139_B2/NP139_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP147_B2/NP147_B2.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP150_B1/NP150_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP153_B1/NP153_B1.nwb"),
    Path("/data_store2/neuropixels/nwb/old/NP171_B1/NP171_B1.nwb"), # new gbm flair-
    Path("/data_store2/neuropixels/nwb/old/NP174_B3/NP174_B3.nwb"), # new gbm flair-
]   

### adjust for the depth of the probe
subj_list = [1, 2, 3, 5, 6, 6, 6, 7, 8, 9, 10, 10, 11, 12, 13, 14, 15, 16, 16, 17, 17, 18, 19, 20, 20, 21, 22, 23]
path_list = ['ast', 'ast', 'gbm','oli', 'gbm', 'gbm', 'gbm', 'gbm', 'gbm', 'ast', 'ast', 'ast', 'ast', 'oli', 'oli', 'gbm', 'ast', 'ast', 'ast', 'ast', 'ast', 'ast', 'gbm', 'ast', 'ast', 'oli', 'gbm', 'gbm']
grade_list = [4, 4, 4, 3, 4, 4, 4, 4, 4, 3, 2, 2, 4, 2, 2, 4, 2, 2, 2, 2, 2, 2, 4, 2, 2, 3, 4, 4]
yield_list = [1, 57, 17, 1, 139, 178, 202, 49, 4, 30, 9, 29, 3, 77, 126, 30, 35, 10, 1, 34, 8, 42, 30, 10, 11, 22, 17, 24]
opercular_list = [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]
region_list = ['aSTG','SFG','aSTG','SFG','vPrCG','vPrCG','vPrCG','pSTG','aMTG','MFG','aSTG','parsOp','MFG','PoCG','PoCG','pSTG','parsOr','vPrCG','pSTG','parsTr','pSTG','parsTr','SMG','parsTr','pSTG','vPrCG', 'aSTG', 'vPrCG']
age_list = [28, 28, 54, 34, 52, 52, 52, 59, 64, 34, 34, 34, 39, 43, 43, 63, 38, 43, 43, 29, 29, 29, 47, 31, 31, 41, 79, 55]
gender_list = [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0]

manual_exclude_lists = [
    [2], # NP93_B1.imec0
    [], # NP95_B1.imec0
    [29, 45, 55], # NP101_B3.imec0
    [130, 150, 151, 245, 371, 375, 377, 380, 386, 452], # NP105_B1.imec0
    [149, 156, 172, 173, 176, 217, 221, 253, 272, 276, 336, 487], # NP113_B1.imec0
    [22, 112, 149, 155, 161, 167, 176, 188, 190, 196, 213, 216, 241, 252, 255, 258, 286, 289, 307, 325, 326, 370, 395, 442, 443, 448, 449, 451, 455, 510, 549, 582, 692, 710, 717, 719, 722, 726, 729], # NP113_B1.imec1
    [385, 448], # NP113_B2.imec2
    [32, 34, 37, 39, 67, 142, 145, 175, 188, 310, 325, 331, 391, 392], # NP114_B1.imec0
    [11, 15, 30, 36], # NP116_B2.imec0
    [340, 382], # NP122_B1.imec0
    [54, 109, 152], # NP128_B1.imec0
    [], # NP128_B1.imec1
    [0, 6, 13, 33, 51, 52], # NP129_B1.imec0
    [19, 62, 111, 151, 171, 192, 199, 200, 205, 210, 227, 229, 244, 290, 298, 311], # NP132_B2.imec0
    [0, 10, 11, 15, 29, 45, 61, 112, 127, 149, 151, 158, 159, 172, 197, 209, 212, 218, 297, 315], # NP132_B3.imec0
    [334, 335, 333], # NP136_B1.imec0
    [361], # NP137_B1.imec0
    [], # NP138_B1.imec0
    [169, 195, 198], # NP138_B1.imec1
    [137, 149, 196, 197, 199, 206, 212, 220, 223, 231], # NP139_B1.imec0
    [129, 248, 249], # NP139_B1.imec1
    [28, 59, 60, 62, 63, 82, 104, 112, 146], # NP139_B2.imec0
    [47, 57, 59, 73, 82, 88, 116, 124, 125, 126, 127, 129, 148, 175, 194, 196, 239, 240, 242, 243, 244, 246, 253, 267, 245, 247, 248, 249, 250, 251, 254], # NP147_B2.imec0
    [], # NP150_B1.imec0
    [34, 64, 119, 135], # NP150_B1.imec1
    [20, 22, 47, 71, 80, 118, 125, 134, 141, 145, 146, 166, 173, 176, 191, 270, 298, 307, 385, 414, 391, 472, 473, 490, 491], # NP153_B1.imec0
    [16, 67, 74, 78, 91, 99, 221, 224, 225, 288], # NP171_B1.imec0
    [9, 11, 13, 23, 40, 43, 47, 59, 67, 72, 87, 104, 109, 122, 166, 169, 172, 175, 197, 199, 203, 205, 211, 212, 219, 227, 229, 230, 235, 237, 240, 251, 256, 295, 296], # NP174_B3.imec0
]

tipDepth_list = [
    5000, # NP93_B1.imec0
    6600, # NP95_B1.imec0
    5843, # NP101_B3.imec0
    7200, # NP105_B1.imec0
    7600, # NP113_B1.imec0
    7600, # NP113_B1.imec1
    7600, # NP113_B1.imec2
    6900, # NP114_B1.imec0
    6100, # NP116_B2.imec0
    6880, # NP122_B1.imec0
    6200, # NP128_B1.imec0
    6200, # NP128_B1.imec1
    6600, # NP129_B1.imec0
    3820, # NP132_B2.imec0
    3820, # NP132_B3.imec0
    6620, # NP136_B1.imec0
    7000, # NP137_B1.imec0
    7100, # NP138_B1.imec0
    7400, # NP138_B1.imec1
    6900, # NP139_B1.imec0
    7400, # NP139_B1.imec1
    7000, # NP139_B2.imec0
    6400, # NP147_B2.imec0
    6840, # NP150_B1.imec0
    6500, # NP150_B1.imec1
    6500, # NP153_B1.imec0
    6800, # NP171_B1.imec0
    5640, # NP174_B3.imec0
]

# Exclude PoCG entries since not using since don't have depth info on them since used dense montage
exclude_indices = [13, 14] #subject definitions for tumor cohort

def exclude_indices_from_list(the_list, indices):
    return [item for idx, item in enumerate(the_list) if idx not in indices]

subj_list = exclude_indices_from_list(subj_list, exclude_indices)
path_list = exclude_indices_from_list(path_list, exclude_indices)
grade_list = exclude_indices_from_list(grade_list, exclude_indices)
yield_list = exclude_indices_from_list(yield_list, exclude_indices)
opercular_list = exclude_indices_from_list(opercular_list, exclude_indices)
region_list = exclude_indices_from_list(region_list, exclude_indices)
age_list = exclude_indices_from_list(age_list, exclude_indices)
gender_list = exclude_indices_from_list(gender_list, exclude_indices)
manual_exclude_lists = exclude_indices_from_list(manual_exclude_lists, exclude_indices)
tipDepth_list = exclude_indices_from_list(tipDepth_list, exclude_indices)

print(f"Loaded {len(nwb_paths)} NWB files")
print(len(subj_list))
print(len(grade_list))
print(len(path_list))
print(len(yield_list))
print(len(opercular_list))
print(len(region_list))
print(len(age_list))
print(len(gender_list))
print(len(manual_exclude_lists))
print(len(tipDepth_list))
print(region_list)
print(subj_list)

In [ ]:
# LOAD SAVED EXTRACTED DATA
# Run this cell instead of re-running the data extraction cell above
# This loads the previously extracted data from extracted_data.pkl

import pickle
from pathlib import Path

load_path = Path('revision_waveform_tumor_v3.pkl')

if load_path.exists():
    print(f"Loading saved data from {load_path}...")
    with open(load_path, 'rb') as f:
        saved_data = pickle.load(f)
    
    # Restore all variables
    indicesAll = saved_data['indicesAll']
    firingRatesAll = saved_data['firingRatesAll']
    waveformAll = saved_data['waveformAll']
    depthAll = saved_data['depthAll']
    burstingMetricsAll = saved_data['burstingMetricsAll']
    autocorrVectorsAll = saved_data['autocorrVectorsAll']
    spikeTimesAll = saved_data['spikeTimesAll']
    insertion = saved_data.get('insertion', len(indicesAll))  # Use saved value or fallback
    
    print(f"✓ Data loaded successfully!")
    print(f"  Processed {insertion} sessions")
    print(f"  Total waveforms: {sum(len(w) for w in waveformAll)}")
    print(f"  Total firing rate measurements: {sum(len(f) for f in firingRatesAll)}")
    print(f"  Total bursting metric measurements: {sum(len(b) for b in burstingMetricsAll)}")
    print(f"  Total spike time measurements: {sum(len(s) for s in spikeTimesAll)}")
    print(f"  File size: {load_path.stat().st_size / (1024**2):.2f} MB")
else:
    print(f"⚠️  Warning: {load_path} not found!")
    print("  Please run the data extraction cell above first to create the saved data file.")
    print("  Or check that the file path is correct.")


In [ ]:
# CLUSTERING ANALYSIS - prepare data > define metrics > collect and standardize > cluster



# 1. prepare data
print("\n=== PREPARING DATA FOR CLUSTERING ===")

# Flatten all data into single arrays
waveforms_flat = []
firing_rates_flat = []
bursting_metrics_flat = []
autocorr_vectors_flat = []
spike_times_flat = []  # NEW: Flatten spike times
metadata_flat = []

for insertion_idx in range(len(waveformAll)):
    for neuron_idx in range(len(waveformAll[insertion_idx])):
        waveforms_flat.append(waveformAll[insertion_idx][neuron_idx])
        firing_rates_flat.append(firingRatesAll[insertion_idx][neuron_idx])
        bursting_metrics_flat.append(burstingMetricsAll[insertion_idx][neuron_idx])
        autocorr_vectors_flat.append(autocorrVectorsAll[insertion_idx][neuron_idx])
        spike_times_flat.append(spikeTimesAll[insertion_idx][neuron_idx])  # NEW
        
        metadata_flat.append({
            'insertion_idx': insertion_idx,
            'unit_id': indicesAll[insertion_idx][neuron_idx],  # Store unit_id for proper matching
            'subj': subj_list[insertion_idx],
            'grade': grade_list[insertion_idx],
            'pathology': path_list[insertion_idx],
            'yield': yield_list[insertion_idx],
            'opercular': opercular_list[insertion_idx],
            'region': region_list[insertion_idx],
            'age': age_list[insertion_idx],
            'gender': gender_list[insertion_idx],
            'depth': depthAll[insertion_idx][neuron_idx]
        })

# Convert to numpy arrays
waveforms_array = np.array(waveforms_flat)
firing_rates_array = np.array(firing_rates_flat)
autocorr_vectors_array = np.array(autocorr_vectors_flat)

print(f"Total neurons: {len(waveforms_flat)}")
print(f"Waveform array shape: {waveforms_array.shape}")
print(f"Firing rates array shape: {firing_rates_array.shape}")
print(f"Autocorr vectors array shape: {autocorr_vectors_array.shape}")
print(f"Spike times data: {len(spike_times_flat)} neurons")  # NEW

# Extract individual bursting metrics
burst_indices = np.array([bm['burst_index'] for bm in bursting_metrics_flat])
refractory_violations = np.array([bm['refractory_violation'] for bm in bursting_metrics_flat])
burst_peak_heights = np.array([bm['burst_peak_height'] for bm in bursting_metrics_flat])
burst_peak_widths = np.array([bm['burst_peak_width'] for bm in bursting_metrics_flat])
autocorr_skewness = np.array([bm['autocorr_skewness'] for bm in bursting_metrics_flat])
short_isi_ratios = np.array([bm['short_isi_ratio'] for bm in bursting_metrics_flat])

print(f"\nBursting metrics extracted:")
print(f"Burst indices: mean={np.mean(burst_indices):.3f}, std={np.std(burst_indices):.3f}")
print(f"Refractory violations: mean={np.mean(refractory_violations):.3f}, std={np.std(refractory_violations):.3f}")
print(f"Burst peak heights: mean={np.mean(burst_peak_heights):.3f}, std={np.std(burst_peak_heights):.3f}")
print(f"Burst peak widths: mean={np.mean(burst_peak_widths):.3f}, std={np.std(burst_peak_widths):.3f}")
print(f"Autocorr skewness: mean={np.mean(autocorr_skewness):.3f}, std={np.std(autocorr_skewness):.3f}")
print(f"Short ISI ratios: mean={np.mean(short_isi_ratios):.3f}, std={np.std(short_isi_ratios):.3f}")

# Standardize waveforms
waveform_scaler = StandardScaler()
waveforms_scaled = waveform_scaler.fit_transform(waveforms_array)
print(f"\nWaveforms standardized: shape={waveforms_scaled.shape}")

# Prepare spiking data (firing rates + bursting metrics)
spiking_data = np.column_stack([
    firing_rates_array,
    burst_indices,
    refractory_violations,
    burst_peak_heights,
    burst_peak_widths,
    autocorr_skewness,
    short_isi_ratios
])

# Standardize spiking data
spiking_scaler = StandardScaler()
spiking_data_scaled = spiking_scaler.fit_transform(spiking_data)
print(f"Spiking data standardized: shape={spiking_data_scaled.shape}")

# Prepare combined data (waveforms + spiking metrics)
# First standardize each component separately, then combine
combined_data = np.column_stack([
    waveforms_scaled,
    spiking_data_scaled
])

print(f"Combined data prepared: shape={combined_data.shape}")
print("Data preparation complete!")



### 2. Define metrics
# COMPREHENSIVE SPIKE CHARACTERISTICS ANALYSIS
# Based on literature review for distinguishing interneurons vs pyramidal neurons

print("\n=== COMPREHENSIVE SPIKE CHARACTERISTICS ANALYSIS ===")
print("Selected metrics based on literature review:")
print("Waveform: Spike Width, Amplitude, Asymmetry, Rise Time, Decay Time")
print("Firing Rate: Mean Firing Rate, Burst Index, ISI CV, ISI Violation Rate, Spike Frequency Adaptation")

# Import additional libraries for comprehensive analysis
from sklearn.mixture import GaussianMixture
from scipy.stats import chi2_contingency, fisher_exact
import matplotlib.patches as mpatches

def calculate_spike_width(waveform):
    """Calculate spike width (trough-to-peak duration)"""
    # Find peak and trough
    peak_idx = np.argmax(waveform)
    trough_idx = np.argmin(waveform)
    
    # Calculate width in samples (assuming 30kHz sampling rate)
    width_samples = abs(peak_idx - trough_idx)
    width_ms = width_samples / 30.0  # Convert to milliseconds
    
    return width_ms

def calculate_spike_amplitude(waveform):
    """Calculate spike amplitude (peak-to-trough amplitude).
    Returns positive if abs(peak) > abs(trough), negative if abs(trough) > abs(peak)."""
    peak_amp = np.max(waveform)
    trough_amp = np.min(waveform)
    amplitude = peak_amp - trough_amp
    if abs(trough_amp) > abs(peak_amp):
        amplitude = -abs(amplitude)
    return amplitude

def calculate_spike_asymmetry(waveform):
    """Calculate spike asymmetry (peak/trough ratio)"""
    peak_amp = np.max(waveform)
    trough_amp = np.min(waveform)
    
    # Avoid division by zero
    if abs(trough_amp) < 1e-10:
        return np.inf
    
    asymmetry = abs(peak_amp / trough_amp)
    return asymmetry

def calculate_spike_rise_time(waveform):
    """Calculate spike rise time (baseline to peak)"""
    # Find baseline (first 10% of waveform)
    baseline_end = len(waveform) // 10
    baseline = np.mean(waveform[:baseline_end])
    
    # Find peak
    peak_idx = np.argmax(waveform)
    
    # Find where waveform crosses baseline going up
    rising_phase = waveform[:peak_idx]
    baseline_crossings = np.where(np.diff(np.sign(rising_phase - baseline)) > 0)[0]
    
    if len(baseline_crossings) > 0:
        rise_start = baseline_crossings[-1]  # Last crossing before peak
        rise_time_samples = peak_idx - rise_start
        rise_time_ms = rise_time_samples / 30.0  # Convert to milliseconds
    else:
        rise_time_ms = peak_idx / 30.0  # Fallback
    
    return rise_time_ms

def calculate_spike_decay_time(waveform):
    """Calculate spike decay time (peak to baseline)"""
    # Find baseline (last 10% of waveform)
    baseline_start = int(len(waveform) * 0.9)
    baseline = np.mean(waveform[baseline_start:])
    
    # Find peak
    peak_idx = np.argmax(waveform)
    
    # Find where waveform crosses baseline going down after peak
    decay_phase = waveform[peak_idx:]
    baseline_crossings = np.where(np.diff(np.sign(decay_phase - baseline)) < 0)[0]
    
    if len(baseline_crossings) > 0:
        decay_end = baseline_crossings[0]  # First crossing after peak
        decay_time_samples = decay_end
        decay_time_ms = decay_time_samples / 30.0  # Convert to milliseconds
    else:
        decay_time_ms = (len(waveform) - peak_idx) / 30.0  # Fallback
    
    return decay_time_ms

def calculate_isi_cv(spike_times):
    """Calculate ISI coefficient of variation"""
    if len(spike_times) < 2:
        return 0
    
    isis = np.diff(spike_times)
    if len(isis) == 0:
        return 0
    
    mean_isi = np.mean(isis)
    if mean_isi == 0:
        return 0
    
    cv = np.std(isis) / mean_isi
    return cv

def calculate_isi_violation_rate(spike_times, refractory_period=0.002):
    """Calculate ISI violation rate (spikes within refractory period)"""
    if len(spike_times) < 2:
        return 0
    
    isis = np.diff(spike_times)
    violations = np.sum(isis < refractory_period)
    total_spikes = len(spike_times)
    
    violation_rate = violations / total_spikes if total_spikes > 0 else 0
    return violation_rate

def calculate_spike_frequency_adaptation(spike_times, window_size=1.0):
    """Calculate spike frequency adaptation"""
    if len(spike_times) < 10:
        return 0
    
    # Divide spike train into windows
    total_time = spike_times[-1] - spike_times[0]
    n_windows = int(total_time / window_size)
    
    if n_windows < 2:
        return 0
    
    window_firing_rates = []
    for i in range(n_windows):
        window_start = spike_times[0] + i * window_size
        window_end = window_start + window_size
        
        spikes_in_window = np.sum((spike_times >= window_start) & (spike_times < window_end))
        firing_rate = spikes_in_window / window_size
        window_firing_rates.append(firing_rate)
    
    if len(window_firing_rates) < 2:
        return 0
    
    # Calculate adaptation as decrease in firing rate over time
    early_rate = np.mean(window_firing_rates[:len(window_firing_rates)//2])
    late_rate = np.mean(window_firing_rates[len(window_firing_rates)//2:])
    
    if early_rate == 0:
        return 0
    
    adaptation = (early_rate - late_rate) / early_rate
    return adaptation

print("Spike characteristic calculation functions defined!")



### 3. Collect and standarize spike metrics
# COLLECT COMPREHENSIVE SPIKE CHARACTERISTICS DATA (CORRECTED VERSION)
print("\n=== COLLECTING COMPREHENSIVE SPIKE CHARACTERISTICS ===")

# Initialize lists for comprehensive characteristics
comprehensive_waveform_metrics = []
comprehensive_firing_rate_metrics = []
comprehensive_metadata = []

print("Processing all neurons for comprehensive spike characteristics...")

# Process each neuron to extract comprehensive metrics
for neuron_idx in range(len(waveforms_flat)):
    if neuron_idx % 100 == 0:
        print(f"Processing neuron {neuron_idx+1}/{len(waveforms_flat)}")
    
    # Get waveform
    waveform = waveforms_flat[neuron_idx]
    
    # Calculate waveform characteristics
    spike_width = calculate_spike_width(waveform)
    spike_amplitude = calculate_spike_amplitude(waveform)
    spike_asymmetry = calculate_spike_asymmetry(waveform)
    spike_rise_time = calculate_spike_rise_time(waveform)
    spike_decay_time = calculate_spike_decay_time(waveform)
    
    waveform_metrics = [spike_width, spike_amplitude, spike_asymmetry, spike_rise_time, spike_decay_time]
    comprehensive_waveform_metrics.append(waveform_metrics)
    
    # Get firing rate characteristics
    firing_rate = firing_rates_flat[neuron_idx]
    burst_index = bursting_metrics_flat[neuron_idx]['burst_index']
    
    # Calculate additional firing rate characteristics using actual spike times
    spike_times_unit = spike_times_flat[neuron_idx]
    
    # Calculate ISI CV from actual spike times
    isi_cv = calculate_isi_cv(spike_times_unit)
    
    # Calculate ISI violation rate from actual spike times
    isi_violation_rate = calculate_isi_violation_rate(spike_times_unit)
    
    # Calculate spike frequency adaptation from actual spike times
    spike_frequency_adaptation = calculate_spike_frequency_adaptation(spike_times_unit)
    
    firing_rate_metrics = [firing_rate, burst_index, isi_cv, isi_violation_rate, spike_frequency_adaptation]
    comprehensive_firing_rate_metrics.append(firing_rate_metrics)
    
    # Store metadata
    comprehensive_metadata.append(metadata_flat[neuron_idx])

# Convert to numpy arrays
comprehensive_waveform_array = np.array(comprehensive_waveform_metrics)
comprehensive_firing_rate_array = np.array(comprehensive_firing_rate_metrics)

print(f"\nComprehensive metrics collected:")
print(f"Waveform metrics shape: {comprehensive_waveform_array.shape}")
print(f"Firing rate metrics shape: {comprehensive_firing_rate_array.shape}")

# Display summary statistics
waveform_metric_names = ['Spike Width (ms)', 'Spike Amplitude', 'Spike Asymmetry', 'Rise Time (ms)', 'Decay Time (ms)']
firing_rate_metric_names = ['Firing Rate (Hz)', 'Burst Index', 'ISI CV', 'ISI Violation Rate', 'Spike Frequency Adaptation']

print("\nWaveform Metrics Summary:")
for i, name in enumerate(waveform_metric_names):
    mean_val = np.mean(comprehensive_waveform_array[:, i])
    std_val = np.std(comprehensive_waveform_array[:, i])
    print(f"  {name}: {mean_val:.3f} ± {std_val:.3f}")

print("\nFiring Rate Metrics Summary:")
for i, name in enumerate(firing_rate_metric_names):
    mean_val = np.mean(comprehensive_firing_rate_array[:, i])
    std_val = np.std(comprehensive_firing_rate_array[:, i])
    print(f"  {name}: {mean_val:.3f} ± {std_val:.3f}")

print("\nComprehensive data collection complete!")

# STANDARDIZE COMPREHENSIVE METRICS AND PREPARE FOR CLUSTERING
print("\n=== STANDARDIZING COMPREHENSIVE METRICS ===")

# Standardize waveform metrics
waveform_scaler_comprehensive = StandardScaler()
comprehensive_waveform_scaled = waveform_scaler_comprehensive.fit_transform(comprehensive_waveform_array)

# Standardize firing rate metrics
firing_rate_scaler_comprehensive = StandardScaler()
comprehensive_firing_rate_scaled = firing_rate_scaler_comprehensive.fit_transform(comprehensive_firing_rate_array)

# Create combined dataset
comprehensive_combined_data = np.column_stack([
    comprehensive_waveform_scaled,
    comprehensive_firing_rate_scaled
])

print(f"Standardized waveform metrics shape: {comprehensive_waveform_scaled.shape}")
print(f"Standardized firing rate metrics shape: {comprehensive_firing_rate_scaled.shape}")
print(f"Combined comprehensive data shape: {comprehensive_combined_data.shape}")



### 4. CLUSTER
# Apply UMAP dimensionality reduction to comprehensive metrics
print("\n=== APPLYING UMAP TO COMPREHENSIVE METRICS ===")

# UMAP parameters
n_neighbors = 15
min_dist = 0.1
n_components = 2
random_state = 42

# UMAP on comprehensive waveform metrics
print("Applying UMAP to comprehensive waveform metrics...")
umap_waveform_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
waveform_comprehensive_umap = umap_waveform_comprehensive.fit_transform(comprehensive_waveform_scaled)

# UMAP on comprehensive firing rate metrics
print("Applying UMAP to comprehensive firing rate metrics...")
umap_firing_rate_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
firing_rate_comprehensive_umap = umap_firing_rate_comprehensive.fit_transform(comprehensive_firing_rate_scaled)

# UMAP on combined comprehensive data
print("Applying UMAP to combined comprehensive data...")
umap_combined_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
combined_comprehensive_umap = umap_combined_comprehensive.fit_transform(comprehensive_combined_data)

print(f"Comprehensive waveform UMAP shape: {waveform_comprehensive_umap.shape}")
print(f"Comprehensive firing rate UMAP shape: {firing_rate_comprehensive_umap.shape}")
print(f"Comprehensive combined UMAP shape: {combined_comprehensive_umap.shape}")

print("UMAP dimensionality reduction complete!")

In [ ]:
# COMPREHENSIVE CLUSTERING ANALYSIS
print("\n=== COMPREHENSIVE CLUSTERING ANALYSIS ===")

# Method 1: UMAP + K-means Clustering
print("\n--- METHOD 1: UMAP + K-means Clustering ---")

def find_optimal_k_comprehensive(data, k_range, method_name):
    """Find optimal number of clusters using silhouette score"""
    silhouette_scores = []
    inertias = []
    
    print(f"Testing optimal k for {method_name}:")
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(data)
        
        silhouette_avg = silhouette_score(data, labels)
        inertia = kmeans.inertia_
        
        silhouette_scores.append(silhouette_avg)
        inertias.append(inertia)
        
        print(f"k={k}: Silhouette Score = {silhouette_avg:.3f}, Inertia = {inertia:.1f}")
    
    optimal_k = k_range[np.argmax(silhouette_scores)]
    print(f"Optimal number of clusters: {optimal_k}")
    
    return optimal_k, silhouette_scores, inertias

# Test different k values
k_range = range(2, 8)

# Find optimal k for each UMAP embedding
optimal_k_waveform_comp, silhouette_scores_waveform_comp, inertias_waveform_comp = find_optimal_k_comprehensive(
    waveform_comprehensive_umap, k_range, "comprehensive waveform UMAP"
)

optimal_k_firing_rate_comp, silhouette_scores_firing_rate_comp, inertias_firing_rate_comp = find_optimal_k_comprehensive(
    firing_rate_comprehensive_umap, k_range, "comprehensive firing rate UMAP"
)

optimal_k_combined_comp, silhouette_scores_combined_comp, inertias_combined_comp = find_optimal_k_comprehensive(
    combined_comprehensive_umap, k_range, "comprehensive combined UMAP"
)

# Perform final clustering with optimal k values
print("\nPerforming final UMAP + K-means clustering...")

# Waveform clustering
kmeans_waveform_comp_final = KMeans(n_clusters=optimal_k_waveform_comp, random_state=42, n_init=10)
waveform_comp_labels = kmeans_waveform_comp_final.fit_predict(waveform_comprehensive_umap)
print(f"Comprehensive waveform clustering completed with {optimal_k_waveform_comp} clusters")

# Firing rate clustering
kmeans_firing_rate_comp_final = KMeans(n_clusters=optimal_k_firing_rate_comp, random_state=42, n_init=10)
firing_rate_comp_labels = kmeans_firing_rate_comp_final.fit_predict(firing_rate_comprehensive_umap)
print(f"Comprehensive firing rate clustering completed with {optimal_k_firing_rate_comp} clusters")

# Combined clustering
kmeans_combined_comp_final = KMeans(n_clusters=optimal_k_combined_comp, random_state=42, n_init=10)
combined_comp_labels = kmeans_combined_comp_final.fit_predict(combined_comprehensive_umap)
print(f"Comprehensive combined clustering completed with {optimal_k_combined_comp} clusters")

print("UMAP + K-means clustering complete!")



# VISUALIZE UMAP CLUSTERS FOR ALL METHODS WITH COLOR KEYS
print("\n=== UMAP CLUSTER VISUALIZATIONS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

# Create figure with subplots for all UMAP embeddings and clustering results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('UMAP Embeddings and K-means Clustering Results (Comprehensive Analysis)', fontsize=16, fontweight='bold')

# Plot 1: Waveform UMAP with clustering
ax1 = axes[0, 0]
# Use plot instead of scatter for cleaner dots without outlines
for cluster_id in range(optimal_k_waveform_comp):
    mask = waveform_comp_labels == cluster_id
    ax1.plot(waveform_comprehensive_umap[mask, 0], waveform_comprehensive_umap[mask, 1], 
             'o', alpha=0.6, markersize=4, markeredgewidth=0, markeredgecolor='none',
             color=plt.cm.tab10(cluster_id / max(1, optimal_k_waveform_comp - 1)),
             label=f'Cluster {cluster_id}')
ax1.set_title(f'Waveform UMAP - K-means Clusters (k={optimal_k_waveform_comp})')
ax1.set_xlabel('UMAP 1')
ax1.set_ylabel('UMAP 2')
ax1.grid(True, alpha=0.3)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 2: Firing Rate UMAP with clustering
ax2 = axes[0, 1]
for cluster_id in range(optimal_k_firing_rate_comp):
    mask = firing_rate_comp_labels == cluster_id
    ax2.plot(firing_rate_comprehensive_umap[mask, 0], firing_rate_comprehensive_umap[mask, 1], 
             'o', alpha=0.6, markersize=4, markeredgewidth=0, markeredgecolor='none',
             color=plt.cm.tab10(cluster_id / max(1, optimal_k_firing_rate_comp - 1)),
             label=f'Cluster {cluster_id}')
ax2.set_title(f'Firing Rate UMAP - K-means Clusters (k={optimal_k_firing_rate_comp})')
ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
ax2.grid(True, alpha=0.3)
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 3: Combined UMAP with clustering
ax3 = axes[0, 2]
for cluster_id in range(optimal_k_combined_comp):
    mask = combined_comp_labels == cluster_id
    ax3.plot(combined_comprehensive_umap[mask, 0], combined_comprehensive_umap[mask, 1], 
             'o', alpha=0.6, markersize=4, markeredgewidth=0, markeredgecolor='none',
             color=plt.cm.tab10(cluster_id / max(1, optimal_k_combined_comp - 1)),
             label=f'Cluster {cluster_id}')
ax3.set_title(f'Combined UMAP - K-means Clusters (k={optimal_k_combined_comp})')
ax3.set_xlabel('UMAP 1')
ax3.set_ylabel('UMAP 2')
ax3.grid(True, alpha=0.3)
ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot 4: Silhouette scores comparison
ax4 = axes[1, 0]
ax4.plot(k_range, silhouette_scores_waveform_comp, 'o-', label='Waveform UMAP', linewidth=2, markersize=6)
ax4.plot(k_range, silhouette_scores_firing_rate_comp, 's-', label='Firing Rate UMAP', linewidth=2, markersize=6)
ax4.plot(k_range, silhouette_scores_combined_comp, '^-', label='Combined UMAP', linewidth=2, markersize=6)
ax4.set_title('Silhouette Scores vs Number of Clusters')
ax4.set_xlabel('Number of Clusters (k)')
ax4.set_ylabel('Silhouette Score')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Plot 5: Elbow curves comparison
ax5 = axes[1, 1]
ax5.plot(k_range, inertias_waveform_comp, 'o-', label='Waveform UMAP', linewidth=2, markersize=6)
ax5.plot(k_range, inertias_firing_rate_comp, 's-', label='Firing Rate UMAP', linewidth=2, markersize=6)
ax5.plot(k_range, inertias_combined_comp, '^-', label='Combined UMAP', linewidth=2, markersize=6)
ax5.set_title('Elbow Curves - Inertia vs Number of Clusters')
ax5.set_xlabel('Number of Clusters (k)')
ax5.set_ylabel('Inertia')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Cluster size comparison
ax6 = axes[1, 2]
cluster_counts = [optimal_k_waveform_comp, optimal_k_firing_rate_comp, optimal_k_combined_comp]
methods = ['Waveform', 'Firing Rate', 'Combined']
colors = ['skyblue', 'lightcoral', 'lightgreen']

bars = ax6.bar(methods, cluster_counts, color=colors, alpha=0.7, edgecolor='black')
ax6.set_title('Optimal Number of Clusters')
ax6.set_ylabel('Number of Clusters')
ax6.set_ylim(0, max(cluster_counts) + 1)

# Add value labels on bars
for bar, count in zip(bars, cluster_counts):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("cluster_results.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Additional detailed cluster analysis
print("\n=== DETAILED CLUSTER ANALYSIS ===")

def analyze_cluster_characteristics_comprehensive(labels, data_name, n_clusters, waveform_data, firing_rate_data):
    """Analyze characteristics of each cluster"""
    print(f"\n{data_name} Cluster Analysis:")
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_size = np.sum(cluster_mask)
        
        if cluster_size > 0:
            # Analyze waveform characteristics for this cluster
            cluster_waveform_metrics = comprehensive_waveform_array[cluster_mask]
            cluster_firing_rate_metrics = comprehensive_firing_rate_array[cluster_mask]
            
            print(f"  Cluster {cluster_id} (n={cluster_size}):")
            
            # Waveform characteristics
            print(f"    Waveform Metrics:")
            print(f"      Spike Width: {np.mean(cluster_waveform_metrics[:, 0]):.3f} ± {np.std(cluster_waveform_metrics[:, 0]):.3f} ms")
            print(f"      Spike Amplitude: {np.mean(cluster_waveform_metrics[:, 1]):.3f} ± {np.std(cluster_waveform_metrics[:, 1]):.3f}")
            print(f"      Spike Asymmetry: {np.mean(cluster_waveform_metrics[:, 2]):.3f} ± {np.std(cluster_waveform_metrics[:, 2]):.3f}")
            print(f"      Rise Time: {np.mean(cluster_waveform_metrics[:, 3]):.3f} ± {np.std(cluster_waveform_metrics[:, 3]):.3f} ms")
            print(f"      Decay Time: {np.mean(cluster_waveform_metrics[:, 4]):.3f} ± {np.std(cluster_waveform_metrics[:, 4]):.3f} ms")
            
            # Firing rate characteristics
            print(f"    Firing Rate Metrics:")
            print(f"      Firing Rate: {np.mean(cluster_firing_rate_metrics[:, 0]):.2f} ± {np.std(cluster_firing_rate_metrics[:, 0]):.2f} Hz")
            print(f"      Burst Index: {np.mean(cluster_firing_rate_metrics[:, 1]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 1]):.3f}")
            print(f"      ISI CV: {np.mean(cluster_firing_rate_metrics[:, 2]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 2]):.3f}")
            print(f"      ISI Violation Rate: {np.mean(cluster_firing_rate_metrics[:, 3]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 3]):.3f}")
            print(f"      Spike Frequency Adaptation: {np.mean(cluster_firing_rate_metrics[:, 4]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 4]):.3f}")

# Analyze each clustering method
analyze_cluster_characteristics_comprehensive(waveform_comp_labels, "Waveform UMAP", optimal_k_waveform_comp, 
                                            comprehensive_waveform_array, comprehensive_firing_rate_array)
analyze_cluster_characteristics_comprehensive(firing_rate_comp_labels, "Firing Rate UMAP", optimal_k_firing_rate_comp, 
                                            comprehensive_waveform_array, comprehensive_firing_rate_array)
analyze_cluster_characteristics_comprehensive(combined_comp_labels, "Combined UMAP", optimal_k_combined_comp, 
                                            comprehensive_waveform_array, comprehensive_firing_rate_array)

print("\nUMAP cluster visualizations complete!")

In [ ]:
# Visualizaiton cell



#  1. DEFINE COMPREHENSIVE VISUALIZATION FUNCTIONS
print("\n=== COMPREHENSIVE VISUALIZATION FUNCTIONS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

def plot_cluster_waveforms_comprehensive(labels, waveforms_array, title_prefix, n_clusters, method_name):
    """Plot average waveforms for each cluster"""
    fig, axes = plt.subplots(1, n_clusters, figsize=(4*n_clusters, 4))
    if n_clusters == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Average Waveforms ({method_name})', fontsize=14, fontweight='bold')
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_waveforms = waveforms_array[cluster_mask]
        
        if len(cluster_waveforms) > 0:
            # Calculate average waveform
            avg_waveform = np.mean(cluster_waveforms, axis=0)
            std_waveform = np.std(cluster_waveforms, axis=0)
            
            # Plot average waveform with error bars
            x_vals = np.arange(len(avg_waveform)) / 30.0  # Convert to ms
            axes[cluster_id].plot(x_vals, avg_waveform, 'b-', linewidth=2, label=f'Cluster {cluster_id}')
            axes[cluster_id].fill_between(x_vals, 
                                        avg_waveform - std_waveform, 
                                        avg_waveform + std_waveform, 
                                        alpha=0.3, color='blue')
            
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_waveforms)})')
            axes[cluster_id].set_xlabel('Time (ms)')
            axes[cluster_id].set_ylabel('Amplitude')
            axes[cluster_id].grid(True, alpha=0.3)
            axes[cluster_id].legend()
    
    plt.tight_layout()
    plt.savefig("cluster_waveforms.svg", format="svg", dpi=300, bbox_inches='tight')
    plt.show()

def plot_region_pie_charts(labels, metadata, title_prefix, n_clusters, method_name):
    """Plot pie charts showing regional distribution for each cluster"""
    fig, axes = plt.subplots(1, n_clusters, figsize=(5*n_clusters, 4))
    if n_clusters == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Regional Distribution ({method_name})', fontsize=14, fontweight='bold')
    
    # Get unique regions
    all_regions = [meta['region'] for meta in metadata]
    unique_regions = list(set(all_regions))
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_metadata = [metadata[i] for i in range(len(metadata)) if cluster_mask[i]]
        
        if len(cluster_metadata) > 0:
            # Count regions in this cluster
            cluster_regions = [meta['region'] for meta in cluster_metadata]
            region_counts = {region: cluster_regions.count(region) for region in unique_regions}
            
            # Create pie chart
            labels_pie = list(region_counts.keys())
            sizes = list(region_counts.values())
            colors = plt.cm.Set3(np.linspace(0, 1, len(labels_pie)))
            
            axes[cluster_id].pie(sizes, labels=labels_pie, autopct='%1.1f%%', colors=colors)
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_metadata)})')
    
    plt.tight_layout()
    plt.savefig("cluster_region.svg", format="svg", dpi=300, bbox_inches='tight')
    plt.show()

print("Comprehensive visualization functions defined!")



# 2. COMPREHENSIVE VISUALIZATIONS FOR UMAP + K-MEANS CLUSTERING
print("\n=== COMPREHENSIVE VISUALIZATIONS: UMAP + K-MEANS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

# Convert waveforms_flat to numpy array for visualization
waveforms_array_for_viz = np.array(waveforms_flat)

# Combined UMAP + K-means visualizations
print("\n--- Combined UMAP + K-means Clustering ---")
plot_cluster_waveforms_comprehensive(combined_comp_labels, waveforms_array_for_viz, 
                                    "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
plot_region_pie_charts(combined_comp_labels, comprehensive_metadata, 
                      "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
#plot_depth_histograms(combined_comp_labels, comprehensive_metadata, 
#                     "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
#plot_subgroup_bar_charts(combined_comp_labels, comprehensive_metadata, 
#                        "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")


In [ ]:
# ET sparse specific visualizations
print("\n=== ET-DENSE vs ET-SPARSE REGIONAL ANALYSIS ===")



# 0. Define ET-dense and ET-sparse regions
et_dense_regions = ['vPrCG', 'PoCG']  # Motor and somatosensory cortex
et_sparse_regions = ['pSTG', 'SFG', 'aSTG', 'aMTG', 'MFG', 'parsOp', 'parsTr', 'parsOr', 'SMG']

print(f"ET-dense regions: {et_dense_regions}")
print(f"ET-sparse regions: {et_sparse_regions}")

# Create masks for ET-dense and ET-sparse regions
et_dense_mask = np.array([neuron['region'] in et_dense_regions for neuron in comprehensive_metadata])
et_sparse_mask = np.array([neuron['region'] in et_sparse_regions for neuron in comprehensive_metadata])

print(f"ET-dense neurons: {np.sum(et_dense_mask)}")
print(f"ET-sparse neurons: {np.sum(et_sparse_mask)}")

# Separate the combined clustering data
et_dense_labels = combined_comp_labels[et_dense_mask]
et_sparse_labels = combined_comp_labels[et_sparse_mask]

et_dense_metadata = [comprehensive_metadata[i] for i in range(len(comprehensive_metadata)) if et_dense_mask[i]]
et_sparse_metadata = [comprehensive_metadata[i] for i in range(len(comprehensive_metadata)) if et_sparse_mask[i]]

et_dense_waveforms = waveforms_array_for_viz[et_dense_mask]
et_sparse_waveforms = waveforms_array_for_viz[et_sparse_mask]



# 1. ET-DENSE REGIONS VISUALIZATION
print("\n=== ET-DENSE REGIONS VISUALIZATION ===")
print(f"Analyzing {len(et_dense_labels)} neurons from ET-dense regions (vPrCG, PoCG)")

# 1. Waveform visualizations for ET-dense
print("\n--- ET-Dense: Waveform Analysis ---")
plot_cluster_waveforms_comprehensive(et_dense_labels, et_dense_waveforms, 
                                    "ET-Dense Combined UMAP", optimal_k_combined_comp, "ET-Dense")

# 2. Region pie charts for ET-dense
print("\n--- ET-Dense: Region Distribution ---")
plot_region_pie_charts(et_dense_labels, et_dense_metadata, 
                      "ET-Dense Combined UMAP", optimal_k_combined_comp, "ET-Dense")



# 2. ET-SPARSE REGIONS VISUALIZATION
print("\n=== ET-SPARSE REGIONS VISUALIZATION ===")
print(f"Analyzing {len(et_sparse_labels)} neurons from ET-sparse regions (all others)")

# 1. Waveform visualizations for ET-sparse
print("\n--- ET-Sparse: Waveform Analysis ---")
plot_cluster_waveforms_comprehensive(et_sparse_labels, et_sparse_waveforms, 
                                    "ET-Sparse Combined UMAP", optimal_k_combined_comp, "ET-Sparse")

# 2. Region pie charts for ET-sparse
print("\n--- ET-Sparse: Region Distribution ---")
plot_region_pie_charts(et_sparse_labels, et_sparse_metadata, 
                      "ET-Sparse Combined UMAP", optimal_k_combined_comp, "ET-Sparse")

In [ ]:
# PIE CHARTS FOR IDH MUTATION STATUS AND GRADE
# SEPARATE ANALYSES FOR OPERCULAR = 0 AND OPERCULAR = 1
plt.rcParams['svg.fonttype'] = 'none' 

# Filter metadata by opercular status
et_sparse_metadata_op0 = [neuron for neuron in et_sparse_metadata if neuron['opercular'] == 0]
et_sparse_metadata_op1 = [neuron for neuron in et_sparse_metadata if neuron['opercular'] == 1]

# Function to calculate counts and percentages
def calculate_stats(metadata_subset):
    # IDH mutation status counts
    idh_mut_count = sum(1 for neuron in metadata_subset if neuron['pathology'] in ['ast', 'oli'])
    idh_wt_count = sum(1 for neuron in metadata_subset if neuron['pathology'] == 'gbm')
    idh_total = idh_mut_count + idh_wt_count
    
    # Grade counts
    grade_2_3_count = sum(1 for neuron in metadata_subset if neuron['grade'] in [2, 3])
    grade_4_count = sum(1 for neuron in metadata_subset if neuron['grade'] == 4)
    grade_total = grade_2_3_count + grade_4_count
    
    # Calculate percentages
    idh_mut_pct = (idh_mut_count / idh_total * 100) if idh_total > 0 else 0
    idh_wt_pct = (idh_wt_count / idh_total * 100) if idh_total > 0 else 0
    
    grade_2_3_pct = (grade_2_3_count / grade_total * 100) if grade_total > 0 else 0
    grade_4_pct = (grade_4_count / grade_total * 100) if grade_total > 0 else 0
    
    return {
        'idh_mut_count': idh_mut_count,
        'idh_wt_count': idh_wt_count,
        'idh_total': idh_total,
        'idh_mut_pct': idh_mut_pct,
        'idh_wt_pct': idh_wt_pct,
        'grade_2_3_count': grade_2_3_count,
        'grade_4_count': grade_4_count,
        'grade_total': grade_total,
        'grade_2_3_pct': grade_2_3_pct,
        'grade_4_pct': grade_4_pct
    }

# Calculate stats for both groups
stats_op0 = calculate_stats(et_sparse_metadata_op0)
stats_op1 = calculate_stats(et_sparse_metadata_op1)

# Create figure with 4 subplots (2 rows x 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
fig.suptitle('ET-Sparse Regions: IDH Status and Grade Distribution by Opercular Status', 
             fontsize=16, fontweight='bold', y=0.995)

# OPERCULAR = 0: IDH Mutation Status
ax1 = axes[0, 0]
idh_labels = ['IDH mut', 'IDH wt']
idh_counts_op0 = [stats_op0['idh_mut_count'], stats_op0['idh_wt_count']]
idh_colors = ['#2ca02c', '#ff7f0e']  # Green for mut, Orange for wt

wedges, texts, autotexts = ax1.pie(idh_counts_op0, labels=idh_labels, colors=idh_colors, autopct='', 
                                   startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, idh_counts_op0, 
                                             [stats_op0['idh_mut_pct'], stats_op0['idh_wt_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax1.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax1.set_title('Opercular = 0: IDH Mutation Status', fontsize=13, fontweight='bold', pad=15)

# OPERCULAR = 0: Grade Distribution
ax2 = axes[0, 1]
grade_labels = ['Grade 2/3', 'Grade 4']
grade_counts_op0 = [stats_op0['grade_2_3_count'], stats_op0['grade_4_count']]
grade_colors = ['#1f77b4', '#d62728']  # Blue for 2/3, Red for 4

wedges, texts, autotexts = ax2.pie(grade_counts_op0, labels=grade_labels, colors=grade_colors, autopct='', 
                                    startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, grade_counts_op0, 
                                             [stats_op0['grade_2_3_pct'], stats_op0['grade_4_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax2.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax2.set_title('Opercular = 0: Grade Distribution', fontsize=13, fontweight='bold', pad=15)

# OPERCULAR = 1: IDH Mutation Status
ax3 = axes[1, 0]
idh_counts_op1 = [stats_op1['idh_mut_count'], stats_op1['idh_wt_count']]

wedges, texts, autotexts = ax3.pie(idh_counts_op1, labels=idh_labels, colors=idh_colors, autopct='', 
                                   startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, idh_counts_op1, 
                                             [stats_op1['idh_mut_pct'], stats_op1['idh_wt_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax3.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax3.set_title('Opercular = 1: IDH Mutation Status', fontsize=13, fontweight='bold', pad=15)

# OPERCULAR = 1: Grade Distribution
ax4 = axes[1, 1]
grade_counts_op1 = [stats_op1['grade_2_3_count'], stats_op1['grade_4_count']]

wedges, texts, autotexts = ax4.pie(grade_counts_op1, labels=grade_labels, colors=grade_colors, autopct='', 
                                    startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

for i, (wedge, count, pct) in enumerate(zip(wedges, grade_counts_op1, 
                                             [stats_op1['grade_2_3_pct'], stats_op1['grade_4_pct']])):
    angle = (wedge.theta2 + wedge.theta1) / 2
    x = np.cos(np.deg2rad(angle))
    y = np.sin(np.deg2rad(angle))
    
    label_text = f'{pct:.1f}%\n(n={count})'
    ax4.text(x * 0.7, y * 0.7, label_text, ha='center', va='center', 
             fontsize=10, fontweight='bold', color='white' if pct > 50 else 'black')

ax4.set_title('Opercular = 1: Grade Distribution', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig("idh_grade_pie_charts_by_opercular.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("=== OPERCULAR = 0 ===")
print("IDH Mutation Status:")
print(f"  IDH mut: {stats_op0['idh_mut_count']} ({stats_op0['idh_mut_pct']:.1f}%)")
print(f"  IDH wt: {stats_op0['idh_wt_count']} ({stats_op0['idh_wt_pct']:.1f}%)")
print(f"  Total: {stats_op0['idh_total']}")
print("\nGrade Distribution:")
print(f"  Grade 2/3: {stats_op0['grade_2_3_count']} ({stats_op0['grade_2_3_pct']:.1f}%)")
print(f"  Grade 4: {stats_op0['grade_4_count']} ({stats_op0['grade_4_pct']:.1f}%)")
print(f"  Total: {stats_op0['grade_total']}")

print("\n=== OPERCULAR = 1 ===")
print("IDH Mutation Status:")
print(f"  IDH mut: {stats_op1['idh_mut_count']} ({stats_op1['idh_mut_pct']:.1f}%)")
print(f"  IDH wt: {stats_op1['idh_wt_count']} ({stats_op1['idh_wt_pct']:.1f}%)")
print(f"  Total: {stats_op1['idh_total']}")
print("\nGrade Distribution:")
print(f"  Grade 2/3: {stats_op1['grade_2_3_count']} ({stats_op1['grade_2_3_pct']:.1f}%)")
print(f"  Grade 4: {stats_op1['grade_4_count']} ({stats_op1['grade_4_pct']:.1f}%)")
print(f"  Total: {stats_op1['grade_total']}")

In [ ]:
# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 0 ONLY
# WITH STATISTICAL COMPARISONS AND POST-HOC PAIRWISE TESTS
plt.rcParams['svg.fonttype'] = 'none' 

# Function to calculate proportions for opercular subgroups
def calculate_proportions_opercular(labels, metadata, group_variable, group_values, opercular_value):
    """Calculate proportion of each cluster within a specific group and opercular status"""
    if isinstance(group_values, str):
        group_values = [group_values]
    
    group_mask = np.array([neuron[group_variable] in group_values for neuron in metadata])
    opercular_mask = np.array([neuron['opercular'] == opercular_value for neuron in metadata])
    combined_mask = group_mask & opercular_mask
    
    group_labels = labels[combined_mask]
    
    if len(group_labels) == 0:
        return np.zeros(3)  # Return zeros if no data
    
    cluster_counts = np.bincount(group_labels, minlength=3)
    proportions = cluster_counts / np.sum(cluster_counts)
    return proportions

# Function to perform chi-square test for cluster proportions
def chi_square_test_clusters(labels, metadata, group_variable, group1_values, group2_values, opercular_value):
    """Perform chi-square test comparing cluster distributions between two groups"""
    from scipy.stats import chi2_contingency
    
    # Get masks for both groups
    group1_mask = np.array([neuron[group_variable] in group1_values for neuron in metadata])
    group2_mask = np.array([neuron[group_variable] in group2_values for neuron in metadata])
    opercular_mask = np.array([neuron['opercular'] == opercular_value for neuron in metadata])
    
    combined_mask1 = group1_mask & opercular_mask
    combined_mask2 = group2_mask & opercular_mask
    
    group1_labels = labels[combined_mask1]
    group2_labels = labels[combined_mask2]
    
    if len(group1_labels) == 0 or len(group2_labels) == 0:
        return None, None
    
    # Create contingency table
    group1_counts = np.bincount(group1_labels, minlength=3)
    group2_counts = np.bincount(group2_labels, minlength=3)
    
    contingency_table = np.array([group1_counts, group2_counts])
    
    # Perform chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    return chi2, p_value

# Function to perform post-hoc pairwise comparisons for individual clusters
def posthoc_cluster_comparisons(labels, metadata, group_variable, group1_values, group2_values, opercular_value):
    """Perform post-hoc comparisons for each individual cluster"""
    from scipy.stats import chi2_contingency
    
    # Get masks for both groups
    group1_mask = np.array([neuron[group_variable] in group1_values for neuron in metadata])
    group2_mask = np.array([neuron[group_variable] in group2_values for neuron in metadata])
    opercular_mask = np.array([neuron['opercular'] == opercular_value for neuron in metadata])
    
    combined_mask1 = group1_mask & opercular_mask
    combined_mask2 = group2_mask & opercular_mask
    
    group1_labels = labels[combined_mask1]
    group2_labels = labels[combined_mask2]
    
    if len(group1_labels) == 0 or len(group2_labels) == 0:
        return None
    
    # Create contingency table
    group1_counts = np.bincount(group1_labels, minlength=3)
    group2_counts = np.bincount(group2_labels, minlength=3)
    
    results = []
    
    # Test each cluster individually (cluster vs all others)
    for cluster_id in range(3):
        # Create 2x2 contingency table: cluster vs all others
        cluster1_count = group1_counts[cluster_id]
        other1_count = np.sum(group1_counts) - cluster1_count
        
        cluster2_count = group2_counts[cluster_id]
        other2_count = np.sum(group2_counts) - cluster2_count
        
        contingency_2x2 = np.array([[cluster1_count, other1_count],
                                   [cluster2_count, other2_count]])
        
        # Perform chi-square test
        chi2, p_value, dof, expected = chi2_contingency(contingency_2x2)
        
        results.append({
            'cluster': cluster_id,
            'chi2': chi2,
            'p_value': p_value,
            'group1_prop': cluster1_count / np.sum(group1_counts),
            'group2_prop': cluster2_count / np.sum(group2_counts)
        })
    
    return results

# Define colors for clusters
cluster_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green

# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 0 ONLY
print("ET-Sparse regions analysis by opercular status = 0:")

# ET-Sparse: Pathology proportions by opercular status = 0
et_sparse_oli_ast_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], 0)
et_sparse_gbm_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['gbm'], 0)

# ET-Sparse: Grade proportions by opercular status = 0
et_sparse_grade2_3_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], 0)
et_sparse_grade4_op0_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [4], 0)

print(f"ET-Sparse oli+ast opercular=0 proportions: {et_sparse_oli_ast_op0_props}")
print(f"ET-Sparse gbm opercular=0 proportions: {et_sparse_gbm_op0_props}")
print(f"ET-Sparse grade 2+3 opercular=0 proportions: {et_sparse_grade2_3_op0_props}")
print(f"ET-Sparse grade 4 opercular=0 proportions: {et_sparse_grade4_op0_props}")

# Perform statistical tests
print("\n=== STATISTICAL TESTS ===")

# Pathology comparison (oli+ast vs gbm)
chi2_path, p_path = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 0)
print(f"Pathology comparison (oli+ast vs gbm):")
print(f"  Overall Chi-square = {chi2_path:.4f}, p-value = {p_path:.4f}")

# Post-hoc pairwise comparisons for pathology
path_posthoc = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 0)
if path_posthoc:
    print("  Post-hoc pairwise comparisons:")
    for result in path_posthoc:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      oli+ast prop = {result['group1_prop']:.3f}, gbm prop = {result['group2_prop']:.3f}")

# Grade comparison (2+3 vs 4)
chi2_grade, p_grade = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 0)
print(f"\nGrade comparison (2+3 vs 4):")
print(f"  Overall Chi-square = {chi2_grade:.4f}, p-value = {p_grade:.4f}")

# Post-hoc pairwise comparisons for grade
grade_posthoc = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 0)
if grade_posthoc:
    print("  Post-hoc pairwise comparisons:")
    for result in grade_posthoc:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      grade 2+3 prop = {result['group1_prop']:.3f}, grade 4 prop = {result['group2_prop']:.3f}")

# Create plots for ET-Sparse regions by opercular status = 0 only
fig_sparse, axes_sparse = plt.subplots(1, 2, figsize=(12, 6))
fig_sparse.suptitle('ET-Sparse Regions: Cluster Proportions (Opercular=0)', fontsize=16, fontweight='bold')

# PLOT 1: ET-Sparse Pathology Proportions - Opercular=0
ax1 = axes_sparse[0]
groups = ['oli+ast', 'gbm']
et_sparse_path_op0_data = [et_sparse_oli_ast_op0_props, et_sparse_gbm_op0_props]

# Create stacked bars
x_pos = np.arange(len(groups))
width = 0.6

bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_path_op0_data).T, cluster_colors)):
    ax1.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax1.set_title('ET-Sparse: Pathology Proportions (Opercular=0)')
ax1.set_ylabel('Proportion')
ax1.set_ylim(0, 1)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(groups)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_path_op0_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax1.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_path is not None:
    significance = "***" if p_path < 0.001 else "**" if p_path < 0.01 else "*" if p_path < 0.05 else "ns"
    ax1.text(0.5, 1.05, f"Overall p = {p_path:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax1.transAxes)

# PLOT 2: ET-Sparse Grade Proportions - Opercular=0
ax2 = axes_sparse[1]
groups = ['Grade 2+3', 'Grade 4']
et_sparse_grade_op0_data = [et_sparse_grade2_3_op0_props, et_sparse_grade4_op0_props]

# Create stacked bars
bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_grade_op0_data).T, cluster_colors)):
    ax2.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax2.set_title('ET-Sparse: Grade Proportions (Opercular=0)')
ax2.set_ylabel('Proportion')
ax2.set_ylim(0, 1)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(groups)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_grade_op0_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax2.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_grade is not None:
    significance = "***" if p_grade < 0.001 else "**" if p_grade < 0.01 else "*" if p_grade < 0.05 else "ns"
    ax2.text(0.5, 1.05, f"Overall p = {p_grade:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax2.transAxes)

plt.tight_layout()
plt.savefig("groupproprotions_sparse.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print detailed cluster-by-cluster comparisons
print("\n=== DETAILED CLUSTER COMPARISONS ===")

# Pathology comparison - cluster by cluster
print("Pathology comparison (oli+ast vs gbm) - Cluster proportions:")
for i in range(3):
    oli_ast_prop = et_sparse_oli_ast_op0_props[i]
    gbm_prop = et_sparse_gbm_op0_props[i]
    diff = gbm_prop - oli_ast_prop


In [ ]:
# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 1 ONLY
# WITH STATISTICAL COMPARISONS AND POST-HOC PAIRWISE TESTS
plt.rcParams['svg.fonttype'] = 'none' 

# ET-SPARSE REGIONS ANALYSIS BY OPERCULAR STATUS = 1 ONLY
print("ET-Sparse regions analysis by opercular status = 1:")

# ET-Sparse: Pathology proportions by opercular status = 1
et_sparse_oli_ast_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], 1)
et_sparse_gbm_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'pathology', ['gbm'], 1)

# ET-Sparse: Grade proportions by opercular status = 1
et_sparse_grade2_3_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], 1)
et_sparse_grade4_op1_props = calculate_proportions_opercular(et_sparse_labels, et_sparse_metadata, 'grade', [4], 1)

print(f"ET-Sparse oli+ast opercular=1 proportions: {et_sparse_oli_ast_op1_props}")
print(f"ET-Sparse gbm opercular=1 proportions: {et_sparse_gbm_op1_props}")
print(f"ET-Sparse grade 2+3 opercular=1 proportions: {et_sparse_grade2_3_op1_props}")
print(f"ET-Sparse grade 4 opercular=1 proportions: {et_sparse_grade4_op1_props}")

# Perform statistical tests
print("\n=== STATISTICAL TESTS ===")

# Pathology comparison (oli+ast vs gbm)
chi2_path_op1, p_path_op1 = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 1)
print(f"Pathology comparison (oli+ast vs gbm):")
print(f"  Overall Chi-square = {chi2_path_op1:.4f}, p-value = {p_path_op1:.4f}")

# Post-hoc pairwise comparisons for pathology
path_posthoc_op1 = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'pathology', ['oli', 'ast'], ['gbm'], 1)
if path_posthoc_op1:
    print("  Post-hoc pairwise comparisons:")
    for result in path_posthoc_op1:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      oli+ast prop = {result['group1_prop']:.3f}, gbm prop = {result['group2_prop']:.3f}")

# Grade comparison (2+3 vs 4)
chi2_grade_op1, p_grade_op1 = chi_square_test_clusters(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 1)
print(f"\nGrade comparison (2+3 vs 4):")
print(f"  Overall Chi-square = {chi2_grade_op1:.4f}, p-value = {p_grade_op1:.4f}")

# Post-hoc pairwise comparisons for grade
grade_posthoc_op1 = posthoc_cluster_comparisons(et_sparse_labels, et_sparse_metadata, 'grade', [2, 3], [4], 1)
if grade_posthoc_op1:
    print("  Post-hoc pairwise comparisons:")
    for result in grade_posthoc_op1:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"    Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"      grade 2+3 prop = {result['group1_prop']:.3f}, grade 4 prop = {result['group2_prop']:.3f}")

# Create plots for ET-Sparse regions by opercular status = 1 only
fig_sparse_op1, axes_sparse_op1 = plt.subplots(1, 2, figsize=(12, 6))
fig_sparse_op1.suptitle('ET-Sparse Regions: Cluster Proportions (Opercular=1)', fontsize=16, fontweight='bold')

# PLOT 1: ET-Sparse Pathology Proportions - Opercular=1
ax1 = axes_sparse_op1[0]
groups = ['oli+ast', 'gbm']
et_sparse_path_op1_data = [et_sparse_oli_ast_op1_props, et_sparse_gbm_op1_props]

# Create stacked bars
x_pos = np.arange(len(groups))
width = 0.6

bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_path_op1_data).T, cluster_colors)):
    ax1.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax1.set_title('ET-Sparse: Pathology Proportions (Opercular=1)')
ax1.set_ylabel('Proportion')
ax1.set_ylim(0, 1)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(groups)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_path_op1_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax1.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_path_op1 is not None:
    significance = "***" if p_path_op1 < 0.001 else "**" if p_path_op1 < 0.01 else "*" if p_path_op1 < 0.05 else "ns"
    ax1.text(0.5, 1.05, f"Overall p = {p_path_op1:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax1.transAxes)

# PLOT 2: ET-Sparse Grade Proportions - Opercular=1
ax2 = axes_sparse_op1[1]
groups = ['Grade 2+3', 'Grade 4']
et_sparse_grade_op1_data = [et_sparse_grade2_3_op1_props, et_sparse_grade4_op1_props]

# Create stacked bars
bottom = np.zeros(len(groups))
for i, (cluster_props, color) in enumerate(zip(np.array(et_sparse_grade_op1_data).T, cluster_colors)):
    ax2.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=f'Cluster {i}' if cluster_props.sum() > 0 else '')
    bottom += cluster_props

ax2.set_title('ET-Sparse: Grade Proportions (Opercular=1)')
ax2.set_ylabel('Proportion')
ax2.set_ylim(0, 1)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(groups)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels at the bottom of each bar
for i, group in enumerate(groups):
    cluster_props = et_sparse_grade_op1_data[i]
    label_text = f"{cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f}"
    ax2.text(i, -0.1, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation between bars
if p_grade_op1 is not None:
    significance = "***" if p_grade_op1 < 0.001 else "**" if p_grade_op1 < 0.01 else "*" if p_grade_op1 < 0.05 else "ns"
    ax2.text(0.5, 1.05, f"Overall p = {p_grade_op1:.4f} {significance}", ha='center', va='bottom', 
             fontweight='bold', fontsize=12, transform=ax2.transAxes)

plt.tight_layout()
plt.savefig("groupproprotions_sparse_op1.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print detailed cluster-by-cluster comparisons
print("\n=== DETAILED CLUSTER COMPARISONS ===")

# Pathology comparison - cluster by cluster
print("Pathology comparison (oli+ast vs gbm) - Cluster proportions:")
for i in range(3):
    oli_ast_prop = et_sparse_oli_ast_op1_props[i]
    gbm_prop = et_sparse_gbm_op1_props[i]
    diff = gbm_prop - oli_ast_prop
    print(f"  Cluster {i}: oli+ast = {oli_ast_prop:.3f}, gbm = {gbm_prop:.3f}, diff = {diff:.3f}")

# Grade comparison - cluster by cluster
print("\nGrade comparison (2+3 vs 4) - Cluster proportions:")
for i in range(3):
    grade2_3_prop = et_sparse_grade2_3_op1_props[i]
    grade4_prop = et_sparse_grade4_op1_props[i]
    diff = grade4_prop - grade2_3_prop
    print(f"  Cluster {i}: grade 2+3 = {grade2_3_prop:.3f}, grade 4 = {grade4_prop:.3f}, diff = {diff:.3f}")

In [ ]:
# ===== OPTION: LOAD YIELD-INDEPENDENT CAPACITY FROM PICKLE FILE =====
# Set this flag to True to use yield-independent capacity from pickle file
# Set to False to calculate capacity locally (original method)
USE_YIELD_INDEPENDENT_CAPACITY = True  # Change to True to load from pickle

capacity_map = {}
if USE_YIELD_INDEPENDENT_CAPACITY:
    import pickle
    import os
    import pandas as pd
    
    print("Loading yield_independent_capacity data...")
    try:
        pkl_path = 'yield_independent_assembly_capacity.pkl'
        if os.path.exists(pkl_path):
            with open(pkl_path, 'rb') as f:
                capacity_data = pickle.load(f)
            
            if isinstance(capacity_data, pd.DataFrame):
                for idx, row in capacity_data.iterrows():
                    insertion_idx = int(row['insertion_idx'])
                    assembly_number = int(row['assembly_number'])
                    capacity = float(row['yield_independent_capacity'])
                    capacity_map[(insertion_idx, assembly_number)] = capacity
            elif isinstance(capacity_data, dict):
                if len(capacity_data) > 0:
                    first_key = list(capacity_data.keys())[0]
                    if isinstance(first_key, tuple) and len(first_key) == 2:
                        capacity_map = capacity_data
                    else:
                        for key, value in capacity_data.items():
                            if isinstance(value, dict) and 'yield_independent_capacity' in value:
                                capacity_map[key] = value['yield_independent_capacity']
            print(f"Loaded from pickle: {len(capacity_map)} entries")
        else:
            csv_path = 'yield_independent_assembly_capacity.csv'
            if os.path.exists(csv_path):
                capacity_df = pd.read_csv(csv_path)
                for idx, row in capacity_df.iterrows():
                    insertion_idx = int(row['insertion_idx'])
                    assembly_number = int(row['assembly_number'])
                    capacity = float(row['yield_independent_capacity'])
                    capacity_map[(insertion_idx, assembly_number)] = capacity
                print(f"Loaded from CSV: {len(capacity_map)} entries")
            else:
                print("WARNING: Neither pickle nor CSV file found. Falling back to local calculation.")
    except Exception as e:
        print(f"ERROR loading capacity data: {e}")
        print("Falling back to local calculation.")

# Mapping function to convert insertion_idx to pickle insertion index
def map_to_pickle_insertion_idx(insertion_counter):
    """Map insertion_counter to pickle insertion_idx"""
    if insertion_counter >= 13:
        return insertion_counter + 2
    else:
        return insertion_counter

In [ ]:
# LOOP THROUGH ET-SPARSE OPERCULAR=0 INSERTIONS FOR E/I BALANCE ANALYSIS
# WITH BOTH ASSEMBLY AND NEURON INFORMATION CAPACITY ANALYSIS
import fnmatch
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import re
import os
from scipy import stats
from sklearn.decomposition import PCA, FastICA
from scipy.linalg import norm
from scipy.stats import kstest

MIN_BEHAVIOR_MINUTES = 0
USE_YIELD_INDEPENDENT_CAPACITY = True 

def _behavior_duration_seconds(task_times_obj):
    starts = np.asarray(task_times_obj.start, dtype=float).ravel()
    ends = np.asarray(task_times_obj.end, dtype=float).ravel()
    if starts.size != ends.size:
        return np.nan
    return float(np.sum(ends - starts))

def _ks_time_bounds_full_recording(spike_times_obj):
    """Min/max time for uniform KS (full recording)."""
    tmin, tmax = np.inf, -np.inf
    for u in range(len(spike_times_obj)):
        idx = spike_times_obj[u].as_series().index.values
        if len(idx):
            tmin = min(tmin, float(np.min(idx)))
            tmax = max(tmax, float(np.max(idx)))
    if np.isfinite(tmin) and np.isfinite(tmax):
        return tmin, tmax
    return np.nan, np.nan

def imec_key_sorter(key):
    """Extract the number after 'imec', default to a large number if not found."""
    match = re.search(r'imec(\d+)', key)
    return int(match.group(1)) if match else float('inf')

def calculate_mutual_information(x, y, bins=20):
    """
    Calculate mutual information between two variables using histogram-based approach.
    """
    valid_mask = ~(np.isnan(x) | np.isnan(y))
    if np.sum(valid_mask) < 2:
        return np.nan

    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    hist_2d, _, _ = np.histogram2d(x_valid, y_valid, bins=bins)
    hist_x = np.sum(hist_2d, axis=1)
    hist_y = np.sum(hist_2d, axis=0)

    p_xy = hist_2d / np.sum(hist_2d)
    p_x = hist_x / np.sum(hist_x)
    p_y = hist_y / np.sum(hist_y)

    mi = 0.0
    for i in range(len(p_x)):
        for j in range(len(p_y)):
            if p_xy[i, j] > 0 and p_x[i] > 0 and p_y[j] > 0:
                mi += p_xy[i, j] * np.log2(p_xy[i, j] / (p_x[i] * p_y[j]))
    return mi

def calculate_information_capacity(data, time_bins, min_samples=10):
    """
    Calculate information storage capacity using multiple metrics.
    """
    valid_mask = ~np.isnan(data)
    if np.sum(valid_mask) < min_samples:
        return {
            'entropy': np.nan,
            'mutual_info_temporal': np.nan,
            'capacity': np.nan,
            'n_valid_samples': np.sum(valid_mask),
            'data_range': np.nan,
            'data_std': np.nan
        }

    valid_data = data[valid_mask]

    n_bins = min(20, len(valid_data) // 5)
    if n_bins < 5:
        n_bins = 5

    hist, _ = np.histogram(valid_data, bins=n_bins)
    prob = hist / np.sum(hist)
    prob = prob[prob > 0]

    if len(prob) < 2:
        entropy_val = np.nan
    else:
        entropy_val = -np.sum(prob * np.log2(prob))

    if len(valid_data) > 1:
        try:
            mi_temporal = calculate_mutual_information(
                valid_data[:-1],
                valid_data[1:],
                bins=min(10, len(valid_data) // 10)
            )
        except Exception:
            mi_temporal = np.nan
    else:
        mi_temporal = np.nan

    if not np.isnan(entropy_val) and not np.isnan(mi_temporal):
        capacity = entropy_val - mi_temporal
    elif not np.isnan(entropy_val):
        capacity = entropy_val
    else:
        capacity = np.nan

    data_range = np.max(valid_data) - np.min(valid_data) if len(valid_data) > 0 else np.nan
    data_std = np.std(valid_data) if len(valid_data) > 0 else np.nan

    return {
        'entropy': entropy_val,
        'mutual_info_temporal': mi_temporal,
        'capacity': capacity,
        'n_valid_samples': np.sum(valid_mask),
        'data_range': data_range,
        'data_std': data_std
    }

def calculate_expression_strength(firingRateMatrix, ap_norm_matrix):
    """
    Calculate expression strength using the method from the paper:
    E(b) = R(b)^T * Oi * R(b)
    """
    n_time_bins, n_neurons = firingRateMatrix.shape
    n_assemblies = ap_norm_matrix.shape[0]
    
    expression_strength = np.zeros((n_time_bins, n_assemblies))
    
    for assembly_idx in range(n_assemblies):
        weight_vector = ap_norm_matrix[assembly_idx, :]
        outer_product = np.outer(weight_vector, weight_vector)
        
        for time_bin in range(n_time_bins):
            firing_rate_vector = firingRateMatrix[time_bin, :]
            expression_strength[time_bin, assembly_idx] = firing_rate_vector.T @ outer_product @ firing_rate_vector
    
    return expression_strength

def calculate_ei_balance(assembly_pattern, cluster_labels):
    """
    Calculate E/I balance for an assembly pattern.
    Multiply by -1 for cluster 1, 0 for cluster 2, +1 for cluster 0.
    """
    ei_balance = 0.0
    for neuron_idx, weight in enumerate(assembly_pattern):
        cluster_id = cluster_labels[neuron_idx]
        if cluster_id == 0:  # Excitatory
            ei_balance += weight * 1
        elif cluster_id == 1:  # Inhibitory
            ei_balance += weight * (-1)
        elif cluster_id == 2:  # Other/Unknown
            ei_balance += weight * 0
    
    return ei_balance 

# Get ET-Sparse opercular=0 insertions
et_sparse_op0_mask = np.array([
    neuron['opercular'] == 0 
    for neuron in et_sparse_metadata
])

et_sparse_op0_neurons = et_sparse_labels[et_sparse_op0_mask]
et_sparse_op0_metadata = [neuron for neuron in et_sparse_metadata if neuron['opercular'] == 0]

# Get unique insertion indices for ET-Sparse opercular=0
et_sparse_op0_insertions = set()
for neuron in et_sparse_op0_metadata:
    et_sparse_op0_insertions.add(neuron['insertion_idx'])

print(f"ET-Sparse opercular=0 insertions: {sorted(et_sparse_op0_insertions)}")
print(f"Total insertions to process: {len(et_sparse_op0_insertions)}")

# Create mapping from nwb_paths index to actual insertion index
# Use the same logic as the original data loading to ensure consistency
nwb_to_insertion_map = {}
insertion_counter = 0

for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = data.keys()

    template = "*imec*"
    keys = [key for key in keys if fnmatch.fnmatch(key, template)]
    ks_keys = [key for key in keys if "KS4" in key]
    if ks_keys:
        keys = ks_keys
    th8_keys = [key for key in keys if "Th=8" in key]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [key for key in keys if "Th=" in key]
        if th_keys:
            keys = th_keys
    template_sentgen = "*sentgen*"
    template_auto = "*_auto*"
    keys = [key for key in keys if not fnmatch.fnmatch(key, template_sentgen) and not fnmatch.fnmatch(key, template_auto)]
    # For NP137, remove 'imec1'
    if 'NP137' in str(nwb_paths[i]) or 'NP139_B2' in str(nwb_paths[i]):
        keys = [key for key in keys if 'imec1' not in key]
    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        nwb_to_insertion_map[(i, s)] = insertion_counter
        insertion_counter += 1

print(f"NWB to insertion mapping created. Total insertions mapped: {insertion_counter}")

# Debug: Print the mapping to see what's happening
print(f"\nDebug: NWB to insertion mapping:")
for (i, s), insertion_idx in nwb_to_insertion_map.items():
    if insertion_idx in et_sparse_op0_insertions:
        print(f"  NWB[{i}], Session[{s}] -> Insertion {insertion_idx}")

# Storage for results
assembly_results = []
neuron_results = []
insertion_results = []

# Process each insertion
for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = data.keys()

    template = "*imec*"
    keys = [key for key in keys if fnmatch.fnmatch(key, template)]
    ks_keys = [key for key in keys if "KS4" in key]
    if ks_keys:
        keys = ks_keys
    th8_keys = [key for key in keys if "Th=8" in key]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [key for key in keys if "Th=" in key]
        if th_keys:
            keys = th_keys
    template_sentgen = "*sentgen*"
    template_auto = "*_auto*"
    keys = [key for key in keys if not fnmatch.fnmatch(key, template_sentgen) and not fnmatch.fnmatch(key, template_auto)]
    # For NP137, remove 'imec1'
    if 'NP137' in str(nwb_paths[i]) or 'NP139_B2' in str(nwb_paths[i]):
        keys = [key for key in keys if 'imec1' not in key]
    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        # Get the correct insertion index using our mapping
        insertion_idx = nwb_to_insertion_map[(i, s)]
        
        # Skip if this insertion is not in ET-Sparse opercular=0
        if insertion_idx not in et_sparse_op0_insertions:
            continue
            
        print(f"\nProcessing insertion {insertion_idx}: {keys[s]}")
        print(f"Region = {region_list[insertion_idx]}")
        print(f"Subject = {subj_list[insertion_idx]}")

        spike_times = data[keys[s]]

        firingRates_all = spike_times.metadata["rate"]
        if "TaskTimes" in data.keys():
            task_times = data["TaskTimes"]
        else:
            task_times = data["task_times"]

        start_time = task_times.start
        end_time = task_times.end
        beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)

        beh_sec = _behavior_duration_seconds(task_times)
        beh_min = beh_sec / 60.0 if np.isfinite(beh_sec) else np.nan
        use_full_recording = (not np.isfinite(beh_min)) or (beh_min < MIN_BEHAVIOR_MINUTES)

        if use_full_recording:
            spike_times_beh = spike_times
            firingRates_beh = firingRates_all
            min_time, max_time = _ks_time_bounds_full_recording(spike_times)
            if not (np.isfinite(min_time) and np.isfinite(max_time)) or (max_time <= min_time):
                print(f"  Short behavior ({beh_min:.2f} min) but could not get recording bounds; skipping shank")
                continue
            print(
                f"  Behavioral duration {beh_min:.2f} min (< {MIN_BEHAVIOR_MINUTES} min) — using full recording for filters; KS window [{min_time:.3f}, {max_time:.3f}] s"
            )
        else:
            spike_times_beh = spike_times.restrict(beh_epochs)
            firingRates_beh = spike_times_beh.metadata["rate"]
            min_time = start_time[0]
            max_time = end_time[-1]
            print(
                f"  Behavioral duration {beh_min:.2f} min (>= {MIN_BEHAVIOR_MINUTES} min) — using TaskTimes restrict; KS window [{min_time:.3f}, {max_time:.3f}] s"
            )

        ks_stats = np.zeros(len(spike_times))
        ks_pvals = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            test = spike_times[u].as_series().index.values
            if len(test) > 1:
                normalized_spike_times = (test - min_time) / (max_time - min_time)
                ks_result = kstest(normalized_spike_times, 'uniform')
                ks_stats[u] = ks_result.statistic
                ks_pvals[u] = ks_result.pvalue
            else:
                ks_stats[u] = np.nan
                ks_pvals[u] = np.nan

        violationThreshold = 3 / 1000
        violationPct = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            unit = spike_times[u]
            unit = unit.as_series().index
            if len(unit) < 100:
                violationPct[u] = 1
            else:
                isi = unit.diff()[1:]
                violations = np.where(isi < violationThreshold)
                violations = np.array(violations)
                violationPct[u] = violations.size / len(isi)

        if "KSLabel" in spike_times.metadata:
            KSLabels = spike_times.metadata["KSLabel"]
        else:
            KSLabels = spike_times.metadata["quality"]
        firingRates = firingRates_beh
        mask1 = violationPct < 3 / 100
        mask2 = firingRates > 0.5
        mask3 = KSLabels != "noise"
        mask4 = ks_stats < 0.3
        mask = mask1 & mask2 & mask3 & mask4
        indicesFinal = firingRates.index[mask]
        # now exclude neurons manually rejected
        indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion_idx])
        # FIX: Ensure indicesFinal is sorted for deterministic matching
        indicesFinal = np.sort(indicesFinal)
        
        spike_times_good = spike_times[indicesFinal]
        print(f"Number of good neurons: {len(spike_times_good)}")

        if len(spike_times_good) == 0:
            print("No good neurons found, skipping insertion")
            continue

        # FIXED: Get cluster labels for this insertion's neurons with deterministic matching
        insertion_cluster_labels = []
        insertion_metadata = []

        # Create a lookup dictionary: unit_id -> (global_idx, cluster_label, metadata)
        # This allows us to match by unit_id instead of position
        unit_id_to_cluster = {}
        for global_idx, global_neuron in enumerate(et_sparse_metadata):
            if (
                global_neuron['insertion_idx'] == insertion_idx
                and global_neuron['opercular'] == 0
            ):
                unit_id = global_neuron.get('unit_id')
                if unit_id is not None:
                    unit_id_to_cluster[unit_id] = {
                        'global_idx': global_idx,
                        'cluster_label': et_sparse_labels[global_idx],
                        'metadata': global_neuron,
                    }

        print(f"Found {len(unit_id_to_cluster)} neurons in global data for insertion {insertion_idx}")

        # Match by unit_id - this is the correct way to match neurons
        for neuron_idx, unit_id in enumerate(indicesFinal):
            if unit_id in unit_id_to_cluster:
                match_info = unit_id_to_cluster[unit_id]
                insertion_cluster_labels.append(match_info['cluster_label'])
                insertion_metadata.append(match_info['metadata'])
            else:
                print(
                    f"Warning: Unit ID {unit_id} not found in global data for insertion {insertion_idx}"
                )
                insertion_cluster_labels.append(-1)  # Unknown cluster
                insertion_metadata.append({})

        insertion_cluster_labels = np.array(insertion_cluster_labels)

        # Debug: Print cluster distribution for this insertion
        cluster_counts = [0, 0, 0]
        for label in insertion_cluster_labels:
            if 0 <= label <= 2:
                cluster_counts[label] += 1

        total_neurons = sum(cluster_counts)
        cluster_props = [count / total_neurons if total_neurons > 0 else 0 for count in cluster_counts]

        print(f"Cluster distribution for insertion {insertion_idx}:")
        print(f"  Cluster 0: {cluster_counts[0]} ({cluster_props[0]:.3f})")
        print(f"  Cluster 1: {cluster_counts[1]} ({cluster_props[1]:.3f})")
        print(f"  Cluster 2: {cluster_counts[2]} ({cluster_props[2]:.3f})")
        print(f"  Total: {total_neurons} neurons")

        timescale = 25 / 1000
        spikeCountMatrix = spike_times_good.count(bin_size=timescale)
        bin_edges = spikeCountMatrix.index.values
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        if len(bin_centers) < len(bin_edges):
            last_width = bin_edges[-1] - bin_edges[-2] if len(bin_edges) > 1 else 0
            last_center = bin_edges[-1] - last_width / 2
            bin_centers = np.append(bin_centers, last_center)
        spikeCountMatrix = spikeCountMatrix.values
        firingRateMatrix = spikeCountMatrix / timescale
        firingRateMatrix = stats.zscore(firingRateMatrix, axis=0)

        pca = PCA()
        firingRateMatrix_pca = pca.fit_transform(firingRateMatrix)
        eigenvalues = pca.explained_variance_
        upperbound = (1 + np.sqrt(firingRateMatrix.shape[1] / firingRateMatrix.shape[0])) ** 2
        assemblyIndices = np.where(eigenvalues > upperbound)
        assemblyEigenvalues = eigenvalues[assemblyIndices]
        print(f"Number of assembly patterns: {len(assemblyEigenvalues)}")

        n_pcs = len(assemblyEigenvalues)
        pc_vectors = pca.components_[:n_pcs, :]

        projections = firingRateMatrix @ pc_vectors.T

        if n_pcs > 0:
            fastica = FastICA(
                n_components=n_pcs,
                algorithm='parallel',
                whiten='unit-variance',
                max_iter=500,
                tol=1e-7,
                random_state=1,
            )
            ica_components = fastica.fit_transform(projections)
            mixing_matrix = fastica.mixing_
            unmixing_matrix = fastica.components_
            ica_assembly_patterns = unmixing_matrix @ pc_vectors

            ap_norm_matrix = ica_assembly_patterns
            ap_norm_member = np.zeros(ica_assembly_patterns.shape)

            for ii in range(n_pcs):
                ap = ica_assembly_patterns[ii, :]
                ap_norm = ap / norm(ap)
                maxW = np.max(ap_norm)
                minW = np.min(ap_norm)
                if abs(minW) > maxW:
                    ap_norm = ap_norm * -1
                ap_norm_matrix[ii, :] = ap_norm
                threshold = np.mean(ap_norm) + np.std(ap_norm) * 2
                for c in range(len(ap)):
                    if ap_norm[c] > threshold:
                        ap_norm_member[ii, c] = 1

            sparsity_list = []
            for ap in range(len(ap_norm_matrix)):
                w = np.array(ap_norm_matrix[ap, :])
                n = w.size
                numerator = np.sqrt(n) - np.sum(np.abs(w))
                denominator = np.sqrt(n) - 1
                sparsity = 1 - numerator / denominator if denominator != 0 else np.nan
                sparsity_list.append(sparsity)

            # --- ASSEMBLY INFORMATION CAPACITY ANALYSIS ---
            insertion_assembly_results = []

            for assembly_idx in range(n_pcs):
                print(f"Analyzing assembly {assembly_idx + 1} of {n_pcs}")
                assembly_pattern = ap_norm_matrix[assembly_idx, :]
                assembly_members = ap_norm_member[assembly_idx, :]

                # Calculate E/I balance
                ei_balance = calculate_ei_balance(assembly_pattern, insertion_cluster_labels)

                # Calculate information capacity
                assembly_expression = ica_components[:, assembly_idx]
                # Calculate or load information capacity
                if USE_YIELD_INDEPENDENT_CAPACITY and capacity_map:
                    # Get yield-independent capacity from loaded data
                    pickle_insertion_idx = map_to_pickle_insertion_idx(insertion_idx)
                    if (pickle_insertion_idx, assembly_idx) in capacity_map:
                        assembly_info_capacity = {'capacity': capacity_map[(pickle_insertion_idx, assembly_idx)]}
                    else:
                        # Fallback to local calculation if not found in map
                        print(f"  Warning: Capacity not found for insertion {pickle_insertion_idx}, assembly {assembly_idx}. Calculating locally.")
                        assembly_info_capacity = calculate_information_capacity(assembly_expression, bin_centers)
                else:
                    # Original method: calculate locally
                    assembly_info_capacity = calculate_information_capacity(assembly_expression, bin_centers)

                neuron_details = []
                for neuron_idx in range(len(assembly_pattern)):
                    if neuron_idx < len(insertion_metadata) and insertion_metadata[neuron_idx]:
                        depth = insertion_metadata[neuron_idx].get('depth', 0.0)
                        metadata = insertion_metadata[neuron_idx]
                    else:
                        depth = 0.0
                        metadata = {}

                    neuron_details.append({
                        'neuron_idx': neuron_idx,
                        'assembly_weight': assembly_pattern[neuron_idx],
                        'is_member': bool(assembly_members[neuron_idx]),
                        'cluster_label': insertion_cluster_labels[neuron_idx],
                        'depth': depth,
                        'metadata': metadata,
                    })

                assembly_result = {
                    'insertion_idx': insertion_idx,
                    'assembly_idx': assembly_idx,
                    'ei_balance': ei_balance,
                    'information_capacity': assembly_info_capacity,
                    'sparsity': sparsity_list[assembly_idx],
                    'n_neurons': len(assembly_pattern),
                    'n_members': np.sum(assembly_members),
                    'neuron_details': neuron_details,
                }

                insertion_assembly_results.append(assembly_result)
                assembly_results.append(assembly_result)

            # --- NEURON INFORMATION CAPACITY ANALYSIS ---
            insertion_neuron_results = []
            
            for neuron_idx in range(firingRateMatrix.shape[1]):
                neuron_activity = firingRateMatrix[:, neuron_idx]
                neuron_info_capacity = calculate_information_capacity(neuron_activity, bin_centers)
                
                # Handle case where metadata might be empty
                if neuron_idx < len(insertion_metadata) and insertion_metadata[neuron_idx]:
                    depth = insertion_metadata[neuron_idx].get('depth', 0.0)
                    metadata = insertion_metadata[neuron_idx]
                else:
                    depth = 0.0
                    metadata = {}
                
                neuron_result = {
                    'insertion_idx': insertion_idx,
                    'neuron_idx': neuron_idx,
                    'cluster_label': insertion_cluster_labels[neuron_idx],
                    'depth': depth,
                    'information_capacity': neuron_info_capacity,
                    'metadata': metadata
                }
                
                insertion_neuron_results.append(neuron_result)
                neuron_results.append(neuron_result)
            
            insertion_result = {
                'insertion_idx': insertion_idx,
                'nwb_path': str(nwb_paths[i]),
                'session_key': keys[s],
                'region': region_list[insertion_idx],
                'subject': subj_list[insertion_idx],
                'pathology': path_list[insertion_idx],
                'grade': grade_list[insertion_idx],
                'n_neurons': len(spike_times_good),
                'n_assemblies': n_pcs,
                'assemblies': insertion_assembly_results,
                'neurons': insertion_neuron_results
            }
            
            insertion_results.append(insertion_result)
            
            print(f"Processed {n_pcs} assemblies and {len(spike_times_good)} neurons for insertion {insertion_idx}")
        else:
            print(f"No assemblies found for insertion {insertion_idx}")

print(f"\nAnalysis complete!")
print(f"Total insertions processed: {len(insertion_results)}")
print(f"Total assemblies analyzed: {len(assembly_results)}")
print(f"Total neurons analyzed: {len(neuron_results)}")

# Print summary
print(f"\n=== SUMMARY ===")
for result in insertion_results:
    print(f"Insertion {result['insertion_idx']}: {result['n_assemblies']} assemblies, "
          f"{result['n_neurons']} neurons, {result['region']}, {result['pathology']}-G{result['grade']}")

# Print E/I balance summary
print(f"\n=== E/I BALANCE SUMMARY ===")
ei_balances = [assembly['ei_balance'] for assembly in assembly_results]
info_capacities_assembly = [assembly['information_capacity']['capacity'] for assembly in assembly_results if not np.isnan(assembly['information_capacity']['capacity'])]

print(f"Assembly E/I Balance: mean = {np.mean(ei_balances):.3f}, std = {np.std(ei_balances):.3f}")
print(f"Assembly Information Capacity: mean = {np.mean(info_capacities_assembly):.3f}, std = {np.std(info_capacities_assembly):.3f}")

# Print neuron information capacity summary
print(f"\n=== NEURON INFORMATION CAPACITY SUMMARY ===")
info_capacities_neuron = [neuron['information_capacity']['capacity'] for neuron in neuron_results if not np.isnan(neuron['information_capacity']['capacity'])]

print(f"Neuron Information Capacity: mean = {np.mean(info_capacities_neuron):.3f}, std = {np.std(info_capacities_neuron):.3f}")

# Print cluster-specific neuron information capacity
print(f"\n=== NEURON INFORMATION CAPACITY BY CLUSTER ===")
for cluster_id in range(3):
    cluster_neurons = [neuron for neuron in neuron_results if neuron['cluster_label'] == cluster_id]
    cluster_capacities = [neuron['information_capacity']['capacity'] for neuron in cluster_neurons if not np.isnan(neuron['information_capacity']['capacity'])]
    
    if len(cluster_capacities) > 0:
        print(f"Cluster {cluster_id}: mean = {np.mean(cluster_capacities):.3f}, std = {np.std(cluster_capacities):.3f}, n = {len(cluster_capacities)}")
    else:
        print(f"Cluster {cluster_id}: no valid data")

In [ ]:
# ENHANCED SIGNIFICANCE TESTING WITH IDH MUTATION STATUS AND COMPONENT INFORMATION
# Focus on ratios + IDH mutation status + component information
# GRADE 4 NEURONS ONLY
plt.rcParams['svg.fonttype'] = 'none' 

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set seaborn style for prettier plots
sns.set_style("whitegrid")
sns.set_palette("husl")

# Create dataset - GRADE 4 ONLY
def create_enhanced_dataset():
    assembly_data = []
    for i, assembly in enumerate(assembly_results):
        insertion_idx = assembly['insertion_idx']
        
        # Only include Grade 4 assemblies
        if grade_list[insertion_idx] != 4:
            continue
        
        # Calculate tone metrics
        weights = []
        cluster_labels = []
        neuron_info_capacities = []
        
        for neuron_detail in assembly['neuron_details']:
            if neuron_detail['metadata']:
                weights.append(neuron_detail['assembly_weight'])
                cluster_labels.append(neuron_detail['cluster_label'])
                
                # Get neuron information capacity from neuron_results
                # Find the corresponding neuron in neuron_results
                neuron_idx = neuron_detail['neuron_idx']
                insertion_idx = assembly['insertion_idx']
                
                # Find the neuron in neuron_results
                neuron_info_capacity = 0.0
                for neuron_result in neuron_results:
                    if (neuron_result['insertion_idx'] == insertion_idx and 
                        neuron_result['neuron_idx'] == neuron_idx):
                        neuron_info_capacity = neuron_result['information_capacity']['capacity']
                        break
                
                neuron_info_capacities.append(neuron_info_capacity)
        
        weights = np.array(weights)
        cluster_labels = np.array(cluster_labels)
        neuron_info_capacities = np.array(neuron_info_capacities)
        
        excitatory_tone = np.sum(weights[cluster_labels == 0])
        inhibitory_tone = np.sum(weights[cluster_labels == 2])
        axonal_tone = np.sum(weights[cluster_labels == 1])
        
        # Calculate ratios
        total_tone = np.sum(np.abs(weights))
        excitatory_ratio = np.abs(excitatory_tone) / (total_tone + 1e-10)
        inhibitory_ratio = np.abs(inhibitory_tone) / (total_tone + 1e-10)
        axonal_ratio = np.abs(axonal_tone) / (total_tone + 1e-10)
        
        # Calculate component information (weighted sum of neuron information capacities)
        component_information = np.sum(weights * neuron_info_capacities)
        
        # Determine IDH mutation status - FIXED LABELING
        pathology = path_list[insertion_idx]
        idh_mutated = 1 if pathology in ['ast', 'oli'] else 0  # IDH-mut = 1, IDH-wt = 0
        
        # Create explicit IDH status labels for clarity
        idh_status = 'IDH-mut' if pathology in ['ast', 'oli'] else 'IDH-wt'
        
        assembly_data.append({
            'assembly_idx': i,
            'insertion_idx': insertion_idx,
            'pathology': pathology,
            'grade': grade_list[insertion_idx],
            'information_capacity': assembly['information_capacity']['capacity'],
            'excitatory_ratio': excitatory_ratio,
            'inhibitory_ratio': inhibitory_ratio,
            'axonal_ratio': axonal_ratio,
            'component_information': component_information,
            'idh_mutated': idh_mutated,
            'idh_status': idh_status  # Add explicit status label
        })
    
    return pd.DataFrame(assembly_data)

df = create_enhanced_dataset()
df_valid = df.dropna(subset=['information_capacity', 'component_information'])

print("=== ENHANCED SIGNIFICANCE TESTING WITH IDH MUTATION STATUS AND COMPONENT INFORMATION ===")
print("=== GRADE 4 NEURONS ONLY ===")
print(f"Sample size: {len(df_valid)} assemblies")

# Check IDH distribution
idh_counts = df_valid['idh_mutated'].value_counts()
print(f"IDH-mutated (ast+oli): {idh_counts.get(1, 0)} assemblies")
print(f"IDH-wildtype (gbm): {idh_counts.get(0, 0)} assemblies")

# Verify IDH labeling
print(f"\n=== IDH LABELING VERIFICATION ===")
print("IDH Status breakdown:")
for status in df_valid['idh_status'].unique():
    subset = df_valid[df_valid['idh_status'] == status]
    print(f"  {status}: {len(subset)} assemblies, avg info capacity: {subset['information_capacity'].mean():.3f}")

# Define features (ratios + IDH mutation status + component information)
features = ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio', 'component_information', 'idh_mutated']
X = df_valid[features].values
y = df_valid['information_capacity'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 1. MULTIPLE REGRESSION MODEL
print(f"\n=== MULTIPLE REGRESSION MODEL ===")
lr = LinearRegression()
lr.fit(X_scaled, y)
y_pred = lr.predict(X_scaled)
r2 = lr.score(X_scaled, y)

print(f"R² = {r2:.3f}")
print(f"Coefficients:")
for i, feature in enumerate(features):
    print(f"  {feature}: {lr.coef_[i]:.3f}")

# 2. PERMUTATION TEST FOR MODEL SIGNIFICANCE
print(f"\n=== PERMUTATION TEST FOR MODEL SIGNIFICANCE ===")

def permutation_test_model(X, y, n_permutations=1000):
    """Test if model R² is significantly better than random"""
    
    # Original model
    lr_orig = LinearRegression()
    lr_orig.fit(X, y)
    r2_orig = lr_orig.score(X, y)
    
    # Permutation test
    r2_permuted = []
    for _ in range(n_permutations):
        y_perm = np.random.permutation(y)
        lr_perm = LinearRegression()
        lr_perm.fit(X, y_perm)
        r2_perm = lr_perm.score(X, y_perm)
        r2_permuted.append(r2_perm)
    
    r2_permuted = np.array(r2_permuted)
    p_value = np.mean(r2_permuted >= r2_orig)
    
    return r2_orig, r2_permuted, p_value

r2_orig, r2_permuted, p_model = permutation_test_model(X_scaled, y)

print(f"Original R² = {r2_orig:.3f}")
print(f"Permuted R² = {np.mean(r2_permuted):.3f} ± {np.std(r2_permuted):.3f}")
print(f"p-value = {p_model:.3f}")
print(f"Model is {'significant' if p_model < 0.05 else 'not significant'}")

# 3. MULTICOLLINEARITY ANALYSIS
print(f"\n=== MULTICOLLINEARITY ANALYSIS ===")

def calculate_vif(X, feature_names):
    """Calculate Variance Inflation Factor for each feature"""
    
    vif_scores = []
    for i in range(len(feature_names)):
        # Regress feature i against all other features
        X_other = np.delete(X, i, axis=1)
        y_feature = X[:, i]
        
        lr_vif = LinearRegression()
        lr_vif.fit(X_other, y_feature)
        r2_vif = lr_vif.score(X_other, y_feature)
        
        vif = 1 / (1 - r2_vif) if r2_vif < 0.99 else float('inf')
        vif_scores.append(vif)
        
        print(f"{feature_names[i]}: VIF = {vif:.2f}")
    
    return vif_scores

vif_scores = calculate_vif(X_scaled, features)

# 4. INDIVIDUAL CORRELATIONS
print(f"\n=== INDIVIDUAL CORRELATIONS ===")
individual_correlations = []
individual_p_values = []
for i, feature in enumerate(features):
    r, p = stats.pearsonr(y, X_scaled[:, i])
    individual_correlations.append(r)
    individual_p_values.append(p)
    print(f"{feature}: r = {r:.3f}, p = {p:.3f}")

# 5. COEFFICIENT SIGNIFICANCE TESTING
print(f"\n=== COEFFICIENT SIGNIFICANCE TESTING ===")

def test_coefficient_significance(X, y, feature_names, n_permutations=1000):
    """Test significance of individual coefficients using permutation"""
    
    # Original model
    lr_orig = LinearRegression()
    lr_orig.fit(X, y)
    coef_orig = lr_orig.coef_
    
    # Permutation test for each coefficient
    coef_permuted = np.zeros((n_permutations, len(feature_names)))
    
    for i in range(n_permutations):
        y_perm = np.random.permutation(y)
        lr_perm = LinearRegression()
        lr_perm.fit(X, y_perm)
        coef_permuted[i] = lr_perm.coef_
    
    # Calculate p-values
    p_values = []
    for i, feature in enumerate(feature_names):
        p_val = np.mean(np.abs(coef_permuted[:, i]) >= np.abs(coef_orig[i]))
        p_values.append(p_val)
        
        print(f"{feature}:")
        print(f"  Coefficient = {coef_orig[i]:.3f}")
        print(f"  p-value = {p_val:.3f}")
        print(f"  {'Significant' if p_val < 0.05 else 'Not significant'}")
        print()
    
    return coef_orig, p_values

coef_orig, coef_p_values = test_coefficient_significance(X_scaled, y, features)

# Convert to numpy arrays for comparison
coef_p_values = np.array(coef_p_values)
vif_scores = np.array(vif_scores)
individual_correlations = np.array(individual_correlations)
individual_p_values = np.array(individual_p_values)

# 6. IDH-SPECIFIC ANALYSIS
print(f"\n=== IDH-SPECIFIC ANALYSIS ===")

# Compare information capacity between IDH groups
idh_mut = df_valid[df_valid['idh_mutated'] == 1]['information_capacity']
idh_wt = df_valid[df_valid['idh_mutated'] == 0]['information_capacity']

# T-test
t_stat, t_p = stats.ttest_ind(idh_mut, idh_wt)
print(f"Information Capacity by IDH Status:")
print(f"  IDH-mutated: {idh_mut.mean():.3f} ± {idh_mut.std():.3f} (n={len(idh_mut)})")
print(f"  IDH-wildtype: {idh_wt.mean():.3f} ± {idh_wt.std():.3f} (n={len(idh_wt)})")
print(f"  T-test: t = {t_stat:.3f}, p = {t_p:.3f}")

# Compare ratios between IDH groups
print(f"\nRatio Comparisons by IDH Status:")
idh_comparisons = {}
for ratio in ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio', 'component_information']:
    mut_vals = df_valid[df_valid['idh_mutated'] == 1][ratio]
    wt_vals = df_valid[df_valid['idh_mutated'] == 0][ratio]
    
    t_stat_ratio, t_p_ratio = stats.ttest_ind(mut_vals, wt_vals)
    idh_comparisons[ratio] = {
        't_stat': t_stat_ratio,
        'p_value': t_p_ratio,
        'mut_mean': mut_vals.mean(),
        'mut_std': mut_vals.std(),
        'wt_mean': wt_vals.mean(),
        'wt_std': wt_vals.std()
    }
    
    print(f"  {ratio}:")
    print(f"    IDH-mut: {mut_vals.mean():.3f} ± {mut_vals.std():.3f}")
    print(f"    IDH-wt: {wt_vals.mean():.3f} ± {wt_vals.std():.3f}")
    print(f"    T-test: t = {t_stat_ratio:.3f}, p = {t_p_ratio:.3f}")

# Create comprehensive visualization with seaborn - FIXED LABELING
fig = plt.figure(figsize=(24, 20))
gs = fig.add_gridspec(5, 4, hspace=0.4, wspace=0.3)

fig.suptitle('ENHANCED SIGNIFICANCE TESTING WITH IDH MUTATION STATUS AND COMPONENT INFORMATION\nGRADE 4 NEURONS ONLY - Ratios + IDH Status + Component Information', 
             fontsize=16, fontweight='bold', y=0.95)

# 1. Correlation plots between ratios and information capacity - FIXED LABELING
ratio_features = ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio']
for i, ratio in enumerate(ratio_features):
    ax = fig.add_subplot(gs[0, i])
    
    # Create scatter plot with explicit IDH status labels
    sns.scatterplot(data=df_valid, x=ratio, y='information_capacity', 
                   hue='idh_status', palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, 
                   alpha=0.7, ax=ax)
    
    # Add regression line
    sns.regplot(data=df_valid, x=ratio, y='information_capacity', 
                scatter=False, color='black', ax=ax)
    
    # Calculate correlation
    r, p = stats.pearsonr(df_valid[ratio], df_valid['information_capacity'])
    
    ax.set_title(f'{ratio.replace("_", " ").title()} vs Information Capacity\nr = {r:.3f}, p = {p:.3f}')
    ax.set_xlabel(ratio.replace('_', ' ').title())
    ax.set_ylabel('Information Capacity')
    ax.legend(title='IDH Status')

# 2. Component information vs information capacity - FIXED LABELING
ax = fig.add_subplot(gs[0, 3])
sns.scatterplot(data=df_valid, x='component_information', y='information_capacity', 
               hue='idh_status', palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, 
               alpha=0.7, ax=ax)
sns.regplot(data=df_valid, x='component_information', y='information_capacity', 
            scatter=False, color='black', ax=ax)

r, p = stats.pearsonr(df_valid['component_information'], df_valid['information_capacity'])
ax.set_title(f'Component Information vs Information Capacity\nr = {r:.3f}, p = {p:.3f}')
ax.set_xlabel('Component Information')
ax.set_ylabel('Information Capacity')
ax.legend(title='IDH Status')

# 3. Correlation plots between ratios - FIXED LABELING
ratio_pairs = [
    ('excitatory_ratio', 'inhibitory_ratio'),
    ('excitatory_ratio', 'axonal_ratio'),
    ('inhibitory_ratio', 'axonal_ratio')
]

for i, (ratio1, ratio2) in enumerate(ratio_pairs):
    ax = fig.add_subplot(gs[1, i])
    
    sns.scatterplot(data=df_valid, x=ratio1, y=ratio2, 
                   hue='idh_status', palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, 
                   alpha=0.7, ax=ax)
    sns.regplot(data=df_valid, x=ratio1, y=ratio2, 
                scatter=False, color='black', ax=ax)
    
    r, p = stats.pearsonr(df_valid[ratio1], df_valid[ratio2])
    ax.set_title(f'{ratio1.replace("_", " ").title()} vs {ratio2.replace("_", " ").title()}\nr = {r:.3f}, p = {p:.3f}')
    ax.set_xlabel(ratio1.replace('_', ' ').title())
    ax.set_ylabel(ratio2.replace('_', ' ').title())
    ax.legend(title='IDH Status')

# 4. Information capacity comparison by IDH status - FIXED LABELING
ax = fig.add_subplot(gs[1, 3])
sns.boxplot(data=df_valid, x='idh_status', y='information_capacity', 
            palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, ax=ax)
sns.stripplot(data=df_valid, x='idh_status', y='information_capacity', 
              color='black', alpha=0.6, size=4, ax=ax)

# Add significance annotation
sig_text = f't = {t_stat:.3f}\np = {t_p:.3f}'
if t_p < 0.001:
    sig_text += '***'
elif t_p < 0.01:
    sig_text += '**'
elif t_p < 0.05:
    sig_text += '*'

ax.set_title(f'Information Capacity by IDH Status\n{sig_text}')
ax.set_xlabel('IDH Status')
ax.set_ylabel('Information Capacity')

# 5. Ratio comparisons by IDH status - FIXED LABELING
for i, ratio in enumerate(ratio_features):
    ax = fig.add_subplot(gs[2, i])
    sns.boxplot(data=df_valid, x='idh_status', y=ratio, 
                palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, ax=ax)
    sns.stripplot(data=df_valid, x='idh_status', y=ratio, 
                  color='black', alpha=0.6, size=4, ax=ax)
    
    # Add significance annotation
    comp = idh_comparisons[ratio]
    sig_text = f't = {comp["t_stat"]:.3f}\np = {comp["p_value"]:.3f}'
    if comp["p_value"] < 0.001:
        sig_text += '***'
    elif comp["p_value"] < 0.01:
        sig_text += '**'
    elif comp["p_value"] < 0.05:
        sig_text += '*'
    
    ax.set_title(f'{ratio.replace("_", " ").title()} by IDH Status\n{sig_text}')
    ax.set_xlabel('IDH Status')
    ax.set_ylabel(ratio.replace('_', ' ').title())

# 6. Component information by IDH status - FIXED LABELING
ax = fig.add_subplot(gs[2, 3])
sns.boxplot(data=df_valid, x='idh_status', y='component_information', 
            palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, ax=ax)
sns.stripplot(data=df_valid, x='idh_status', y='component_information', 
              color='black', alpha=0.6, size=4, ax=ax)

# Add significance annotation
comp = idh_comparisons['component_information']
sig_text = f't = {comp["t_stat"]:.3f}\np = {comp["p_value"]:.3f}'
if comp["p_value"] < 0.001:
    sig_text += '***'
elif comp["p_value"] < 0.01:
    sig_text += '**'
elif comp["p_value"] < 0.05:
    sig_text += '*'

ax.set_title(f'Component Information by IDH Status\n{sig_text}')
ax.set_xlabel('IDH Status')
ax.set_ylabel('Component Information')

# 7. Model R² vs permutations
ax = fig.add_subplot(gs[3, 0])
sns.histplot(r2_permuted, bins=30, alpha=0.7, color='blue', ax=ax)
ax.axvline(r2_orig, color='red', linestyle='--', linewidth=2, label=f'Original R² = {r2_orig:.3f}')
ax.set_xlabel('R²')
ax.set_ylabel('Density')
ax.set_title('Model R² Permutation Test')
ax.legend()
ax.grid(True, alpha=0.3)

# 8. Coefficient significance
ax = fig.add_subplot(gs[3, 1])
colors = ['red' if p < 0.05 else 'blue' for p in coef_p_values]
bars = ax.bar(range(len(features)), coef_orig, color=colors, alpha=0.7)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Significance\n(Red = p < 0.05)')
ax.grid(True, alpha=0.3)

# 9. VIF scores
ax = fig.add_subplot(gs[3, 2])
colors_vif = ['red' if vif > 10 else 'orange' if vif > 5 else 'green' for vif in vif_scores]
bars = ax.bar(range(len(features)), vif_scores, color=colors_vif, alpha=0.7)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_ylabel('VIF Score')
ax.set_title('Variance Inflation Factor\n(Red = High Multicollinearity)')
ax.axhline(y=10, color='red', linestyle='--', alpha=0.8, label='VIF = 10')
ax.axhline(y=5, color='orange', linestyle='--', alpha=0.8, label='VIF = 5')
ax.legend()
ax.grid(True, alpha=0.3)

# 10. Individual correlations
ax = fig.add_subplot(gs[3, 3])
colors_corr = ['red' if abs(r) > 0.5 else 'orange' if abs(r) > 0.3 else 'blue' for r in individual_correlations]
bars = ax.bar(range(len(features)), individual_correlations, color=colors_corr, alpha=0.7)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_ylabel('Correlation Coefficient')
ax.set_title('Individual Correlations\n(Red = Large Effect)')
ax.grid(True, alpha=0.3)

# 11. ENHANCED Feature correlation matrix with significance indicators
ax = fig.add_subplot(gs[4, :2])
corr_matrix = np.corrcoef(X_scaled.T)

# Create cleaner feature names for display
feature_names_display = ['Excitatory\nRatio', 'Inhibitory\nRatio', 'Axonal\nRatio', 'Component\nInformation', 'IDH\nMutated']

# Calculate significance for each correlation
sig_matrix = np.zeros_like(corr_matrix, dtype=bool)
for i in range(len(features)):
    for j in range(len(features)):
        if i != j:
            _, p_val = stats.pearsonr(X_scaled[:, i], X_scaled[:, j])
            if p_val < 0.05:
                sig_matrix[i, j] = True

# Plot correlation matrix with all values shown
im = sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
                 square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax,
                 xticklabels=feature_names_display, yticklabels=feature_names_display,
                 fmt='.2f', annot_kws={'fontsize': 10})

# Add significance indicators
for i in range(len(features)):
    for j in range(len(features)):
        if i != j and sig_matrix[i, j]:
            # Add asterisk for significant correlations
            ax.text(j + 0.5, i + 0.7, '*', ha='center', va='center', 
                   fontsize=14, fontweight='bold', color='white')

ax.set_title('Feature Correlation Matrix\n(* = p < 0.05)', fontsize=12, fontweight='bold')

# 12. Summary statistics
ax = fig.add_subplot(gs[4, 2:])
ax.text(0.05, 0.9, 'ENHANCED ANALYSIS SUMMARY (GRADE 4 ONLY)', fontsize=14, fontweight='bold')

# Model statistics
ax.text(0.05, 0.8, 'Model Performance:', fontsize=12, fontweight='bold')
ax.text(0.05, 0.75, f'• Sample size: {len(df_valid)} assemblies', fontsize=11)
ax.text(0.05, 0.7, f'• Model R²: {r2_orig:.3f}', fontsize=11)

# Fix the f-string issue by using a variable
significance_text = 'significant' if p_model < 0.05 else 'not significant'
ax.text(0.05, 0.65, f'• Model p-value: {p_model:.3f} ({significance_text})', fontsize=11)

# Feature analysis
ax.text(0.35, 0.8, 'Feature Analysis:', fontsize=12, fontweight='bold')
ax.text(0.35, 0.75, f'• {sum(coef_p_values < 0.05)}/{len(features)} coefficients significant', fontsize=11)
ax.text(0.35, 0.7, f'• {sum(vif_scores > 10)}/{len(features)} features have high VIF (>10)', fontsize=11)
ax.text(0.35, 0.65, f'• {sum(vif_scores > 5)}/{len(features)} features have moderate VIF (>5)', fontsize=11)

# Key findings
ax.text(0.65, 0.8, 'Key Findings:', fontsize=12, fontweight='bold')
for i, feature in enumerate(features):
    ax.text(0.65, 0.75 - i*0.05, f'• {feature}: r = {individual_correlations[i]:.3f}', fontsize=11)

# Component information interpretation
ax.text(0.05, 0.5, 'Component Information Interpretation:', fontsize=12, fontweight='bold')
ax.text(0.05, 0.45, '1. Weighted sum of neuron information capacities', fontsize=11)
ax.text(0.05, 0.4, '2. Assembly pattern × neuron information vector', fontsize=11)
ax.text(0.05, 0.35, '3. Captures information-weighted circuit organization', fontsize=11)
ax.text(0.05, 0.3, '4. Tests if information-rich neurons drive assemblies', fontsize=11)

# Recommendations
ax.text(0.35, 0.5, 'Recommendations:', fontsize=12, fontweight='bold')
ax.text(0.35, 0.45, '1. Compare component vs assembly information', fontsize=11)
ax.text(0.35, 0.4, '2. Test if information-rich neurons are key', fontsize=11)
ax.text(0.35, 0.35, '3. Validate weighted information hypothesis', fontsize=11)
ax.text(0.35, 0.3, '4. Focus on information-weighted circuit effects', fontsize=11)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.savefig("tone_model_fixed_labeling.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n=== ENHANCED ANALYSIS SUMMARY (GRADE 4 ONLY) ===")
print("This analysis includes component information for Grade 4 neurons only:")
print("1. Do ratios + component information predict capacity in Grade 4?")
print("2. Is component information a key predictor in high-grade tumors?")
print("3. Do information-rich neurons drive assemblies in Grade 4?")
print("4. Test information-weighted circuit organization in high-grade tumors")
print("5. Validate weighted information hypothesis in Grade 4 context")

In [ ]:
# STANDARD OLS COEFFICIENT SIGNIFICANCE TESTING
# Compare permutation test results with standard OLS statistics
print(f"\n{'='*80}")
print("STANDARD OLS COEFFICIENT SIGNIFICANCE TESTING")
print(f"{'='*80}")

import statsmodels.api as sm

# Add constant term for statsmodels (intercept)
X_with_const = sm.add_constant(X_scaled)

# Fit OLS model using statsmodels
ols_model = sm.OLS(y, X_with_const).fit()

# Get summary statistics
print(f"\nModel Summary:")
print(f"  R² = {ols_model.rsquared:.6f}")
print(f"  Adjusted R² = {ols_model.rsquared_adj:.6f}")
print(f"  F-statistic = {ols_model.fvalue:.4f}")
print(f"  F p-value = {ols_model.f_pvalue:.6f}")
print(f"  Degrees of freedom (model) = {ols_model.df_model:.0f}")
print(f"  Degrees of freedom (residual) = {ols_model.df_resid:.0f}")
print(f"  Total degrees of freedom = {ols_model.df_model + ols_model.df_resid:.0f}")

# Extract coefficient statistics (excluding intercept which is at index 0)
print(f"\n{'='*80}")
print("COEFFICIENT STATISTICS COMPARISON")
print(f"{'='*80}")
print(f"\n{'Feature':<25} {'Coefficient':<12} {'Std Error':<12} {'t-stat':<10} {'df':<6} {'p (OLS)':<12} {'p (Perm)':<12} {'Sig (OLS)':<10} {'Sig (Perm)':<10}")
print("-" * 120)

# Compare permutation and OLS results
for i, feature in enumerate(features):
    # OLS statistics (coefficient index is i+1 because intercept is at index 0)
    coef_ols = ols_model.params[i+1]
    std_err_ols = ols_model.bse[i+1]
    t_stat_ols = ols_model.tvalues[i+1]
    p_val_ols = ols_model.pvalues[i+1]
    df_ols = ols_model.df_resid
    
    # Permutation statistics
    coef_perm = coef_orig[i]
    p_val_perm = coef_p_values[i]
    
    # Significance markers
    sig_ols = '***' if p_val_ols < 0.001 else '**' if p_val_ols < 0.01 else '*' if p_val_ols < 0.05 else 'ns'
    sig_perm = '***' if p_val_perm < 0.001 else '**' if p_val_perm < 0.01 else '*' if p_val_perm < 0.05 else 'ns'
    
    print(f"{feature:<25} {coef_ols:>11.6f} {std_err_ols:>11.6f} {t_stat_ols:>9.4f} {df_ols:>5.0f} {p_val_ols:>11.6f} {p_val_perm:>11.6f} {sig_ols:>9} {sig_perm:>9}")

# Intercept statistics
print(f"\n{'Intercept':<25} {ols_model.params[0]:>11.6f} {ols_model.bse[0]:>11.6f} {ols_model.tvalues[0]:>9.4f} {ols_model.df_resid:>5.0f} {ols_model.pvalues[0]:>11.6f} {'N/A':>11} {'***' if ols_model.pvalues[0] < 0.001 else '**' if ols_model.pvalues[0] < 0.01 else '*' if ols_model.pvalues[0] < 0.05 else 'ns':>9} {'N/A':>9}")

print(f"\n{'='*80}")
print("NOTES:")
print(f"{'='*80}")
print("  - OLS p-values: Two-tailed t-test p-values from standard OLS regression")
print("  - Perm p-values: Permutation test p-values (one-tailed, based on |coef|)")
print("  - df: Degrees of freedom for t-test (residual df)")
print("  - Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print(f"\n  Model degrees of freedom: {ols_model.df_model:.0f}")
print(f"  Residual degrees of freedom: {ols_model.df_resid:.0f}")
print(f"  Total sample size: {len(y)}")
print(f"  Number of predictors: {len(features)}")


In [ ]:
# ENHANCED MODEL VISUALIZATION WITH PREDICTED VS ACTUAL ANALYSIS
# Building off the previous analysis with beautiful visualizations
plt.rcParams['svg.fonttype'] = 'none' 

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set seaborn style for prettier plots
sns.set_style("whitegrid")
sns.set_palette("husl")

# Create dataset - GRADE 4 ONLY (same as before)
def create_enhanced_dataset():
    assembly_data = []
    for i, assembly in enumerate(assembly_results):
        insertion_idx = assembly['insertion_idx']
        
        # Only include Grade 4 assemblies
        if grade_list[insertion_idx] != 4:
            continue
        
        # Calculate tone metrics
        weights = []
        cluster_labels = []
        neuron_info_capacities = []
        
        for neuron_detail in assembly['neuron_details']:
            if neuron_detail['metadata']:
                weights.append(neuron_detail['assembly_weight'])
                cluster_labels.append(neuron_detail['cluster_label'])
                
                # Get neuron information capacity from neuron_results
                neuron_idx = neuron_detail['neuron_idx']
                insertion_idx = assembly['insertion_idx']
                
                # Find the neuron in neuron_results
                neuron_info_capacity = 0.0
                for neuron_result in neuron_results:
                    if (neuron_result['insertion_idx'] == insertion_idx and 
                        neuron_result['neuron_idx'] == neuron_idx):
                        neuron_info_capacity = neuron_result['information_capacity']['capacity']
                        break
                
                neuron_info_capacities.append(neuron_info_capacity)
        
        weights = np.array(weights)
        cluster_labels = np.array(cluster_labels)
        neuron_info_capacities = np.array(neuron_info_capacities)
        
        excitatory_tone = np.sum(weights[cluster_labels == 0])
        inhibitory_tone = np.sum(weights[cluster_labels == 2])
        axonal_tone = np.sum(weights[cluster_labels == 1])
        
        # Calculate ratios
        total_tone = np.sum(np.abs(weights))
        excitatory_ratio = np.abs(excitatory_tone) / (total_tone + 1e-10)
        inhibitory_ratio = np.abs(inhibitory_tone) / (total_tone + 1e-10)
        axonal_ratio = np.abs(axonal_tone) / (total_tone + 1e-10)
        
        # Calculate component information (weighted sum of neuron information capacities)
        component_information = np.sum(weights * neuron_info_capacities)
        
        # Determine IDH mutation status
        pathology = path_list[insertion_idx]
        idh_mutated = 1 if pathology in ['ast', 'oli'] else 0
        
        assembly_data.append({
            'assembly_idx': i,
            'insertion_idx': insertion_idx,
            'pathology': pathology,
            'grade': grade_list[insertion_idx],
            'information_capacity': assembly['information_capacity']['capacity'],
            'excitatory_ratio': excitatory_ratio,
            'inhibitory_ratio': inhibitory_ratio,
            'axonal_ratio': axonal_ratio,
            'component_information': component_information,
            'idh_mutated': idh_mutated
        })
    
    return pd.DataFrame(assembly_data)

df = create_enhanced_dataset()
df_valid = df.dropna(subset=['information_capacity', 'component_information'])

print("=== ENHANCED MODEL VISUALIZATION WITH PREDICTED VS ACTUAL ANALYSIS ===")
print("=== GRADE 4 NEURONS ONLY ===")
print(f"Sample size: {len(df_valid)} assemblies")

# Define features and prepare data
features = ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio', 'component_information', 'idh_mutated']
X = df_valid[features].values
y = df_valid['information_capacity'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Build the model
lr = LinearRegression()
lr.fit(X_scaled, y)
y_pred = lr.predict(X_scaled)

# Calculate model metrics
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mse)

print(f"Model Performance:")
print(f"  R² = {r2:.3f}")
print(f"  RMSE = {rmse:.3f}")
print(f"  MAE = {mae:.3f}")

# Add predictions to dataframe for plotting
df_valid['predicted_capacity'] = y_pred
df_valid['residuals'] = y - y_pred

# Create beautiful comprehensive visualization
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)

fig.suptitle('ENHANCED MODEL VISUALIZATION: PREDICTED VS ACTUAL ANALYSIS\nGRADE 4 NEURONS ONLY - Multiple Regression Model', 
             fontsize=18, fontweight='bold', y=0.95)

# 1. PREDICTED VS ACTUAL - Main plot
ax1 = fig.add_subplot(gs[0, :2])

# Create scatter plot with IDH status coloring
scatter = ax1.scatter(y, y_pred, c=df_valid['idh_mutated'], 
                     cmap='RdYlBu', alpha=0.7, s=80, edgecolors='black', linewidth=0.5)

# Add perfect prediction line
min_val = min(y.min(), y_pred.min())
max_val = max(y.max(), y_pred.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, alpha=0.8, label='Perfect Prediction')

# Add regression line
z = np.polyfit(y, y_pred, 1)
p = np.poly1d(z)
ax1.plot(y, p(y), 'r-', linewidth=2, alpha=0.8, label=f'Regression Line (slope={z[0]:.3f})')

# Calculate correlation between actual and predicted
corr_coef, corr_p = stats.pearsonr(y, y_pred)

ax1.set_xlabel('Actual Information Capacity', fontsize=12, fontweight='bold')
ax1.set_ylabel('Predicted Information Capacity', fontsize=12, fontweight='bold')
ax1.set_title(f'Predicted vs Actual Information Capacity\nR² = {r2:.3f}, RMSE = {rmse:.3f}, r = {corr_coef:.3f}', 
              fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('IDH Status (0=IDH-wt, 1=IDH-mut)', fontsize=10)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['IDH-wt', 'IDH-mut'])

ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. RESIDUALS PLOT
ax2 = fig.add_subplot(gs[0, 2:])

# Residuals vs predicted
ax2.scatter(y_pred, df_valid['residuals'], c=df_valid['idh_mutated'], 
           cmap='RdYlBu', alpha=0.7, s=80, edgecolors='black', linewidth=0.5)

# Add horizontal line at zero
ax2.axhline(y=0, color='k', linestyle='--', linewidth=2, alpha=0.8)

# Add regression line for residuals
z_res = np.polyfit(y_pred, df_valid['residuals'], 1)
p_res = np.poly1d(z_res)
ax2.plot(y_pred, p_res(y_pred), 'r-', linewidth=2, alpha=0.8)

ax2.set_xlabel('Predicted Information Capacity', fontsize=12, fontweight='bold')
ax2.set_ylabel('Residuals (Actual - Predicted)', fontsize=12, fontweight='bold')
ax2.set_title('Residuals vs Predicted Values\n(Homogeneity Check)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. FEATURE IMPORTANCE - Standardized Coefficients
ax3 = fig.add_subplot(gs[1, :2])

# Calculate standardized coefficients
feature_names_display = ['Excitatory\nRatio', 'Inhibitory\nRatio', 'Axonal\nRatio', 
                        'Component\nInformation', 'IDH\nMutated']
std_coefs = lr.coef_

# Color bars based on significance (you can add significance testing here)
colors = ['#2E8B57' if abs(coef) > 0.1 else '#4682B4' for coef in std_coefs]
bars = ax3.bar(range(len(feature_names_display)), std_coefs, color=colors, alpha=0.8, 
               edgecolor='black', linewidth=1)

# Add value labels on bars
for i, (bar, coef) in enumerate(zip(bars, std_coefs)):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.03),
             f'{coef:.3f}', ha='center', va='bottom' if height >= 0 else 'top', 
             fontweight='bold', fontsize=10)

ax3.set_xticks(range(len(feature_names_display)))
ax3.set_xticklabels(feature_names_display, fontsize=11)
ax3.set_ylabel('Standardized Coefficient', fontsize=12, fontweight='bold')
ax3.set_title('Feature Importance (Standardized Coefficients)', fontsize=14, fontweight='bold')
ax3.axhline(y=0, color='k', linestyle='-', linewidth=1, alpha=0.5)
ax3.grid(True, alpha=0.3)

# 4. MODEL PERFORMANCE BY IDH STATUS
ax4 = fig.add_subplot(gs[1, 2:])

# Calculate R² for each IDH group
idh_wt_mask = df_valid['idh_mutated'] == 0
idh_mut_mask = df_valid['idh_mutated'] == 1

r2_wt = r2_score(df_valid[idh_wt_mask]['information_capacity'], 
                 df_valid[idh_wt_mask]['predicted_capacity'])
r2_mut = r2_score(df_valid[idh_mut_mask]['information_capacity'], 
                  df_valid[idh_mut_mask]['predicted_capacity'])

# Create bar plot
groups = ['IDH-wt', 'IDH-mut']
r2_values = [r2_wt, r2_mut]
colors_perf = ['#FF6B6B', '#4ECDC4']

bars = ax4.bar(groups, r2_values, color=colors_perf, alpha=0.8, 
               edgecolor='black', linewidth=1)

# Add value labels
for bar, r2_val in zip(bars, r2_values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'R² = {r2_val:.3f}', ha='center', va='bottom', 
             fontweight='bold', fontsize=12)

ax4.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax4.set_title('Model Performance by IDH Status', fontsize=14, fontweight='bold')
ax4.set_ylim(0, max(r2_values) * 1.2)
ax4.grid(True, alpha=0.3)

# 5. PREDICTION ACCURACY DISTRIBUTION
ax5 = fig.add_subplot(gs[2, :2])

# Calculate prediction errors
prediction_errors = np.abs(df_valid['residuals'])
mae_by_idh = [prediction_errors[idh_wt_mask].mean(), prediction_errors[idh_mut_mask].mean()]

# Create histogram of prediction errors
ax5.hist(prediction_errors[idh_wt_mask], bins=15, alpha=0.7, color='#FF6B6B', 
         label=f'IDH-wt (MAE={mae_by_idh[0]:.3f})', density=True)
ax5.hist(prediction_errors[idh_mut_mask], bins=15, alpha=0.7, color='#4ECDC4', 
         label=f'IDH-mut (MAE={mae_by_idh[1]:.3f})', density=True)

ax5.set_xlabel('Absolute Prediction Error', fontsize=12, fontweight='bold')
ax5.set_ylabel('Density', fontsize=12, fontweight='bold')
ax5.set_title('Distribution of Prediction Errors by IDH Status', fontsize=14, fontweight='bold')
ax5.legend(fontsize=11)
ax5.grid(True, alpha=0.3)

# 6. FEATURE CORRELATION HEATMAP
ax6 = fig.add_subplot(gs[2, 2:])

# Calculate correlation matrix
corr_matrix = np.corrcoef(X_scaled.T)

# Create heatmap
im = sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
                 square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax6,
                 xticklabels=feature_names_display, yticklabels=feature_names_display,
                 fmt='.2f', annot_kws={'fontsize': 10})

ax6.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

# 7. COMPONENT INFORMATION ANALYSIS
ax7 = fig.add_subplot(gs[3, :2])

# Scatter plot of component information vs actual information capacity
scatter = ax7.scatter(df_valid['component_information'], df_valid['information_capacity'], 
                     c=df_valid['idh_mutated'], cmap='RdYlBu', alpha=0.7, s=80, 
                     edgecolors='black', linewidth=0.5)

# Add regression line
z_comp = np.polyfit(df_valid['component_information'], df_valid['information_capacity'], 1)
p_comp = np.poly1d(z_comp)
ax7.plot(df_valid['component_information'], p_comp(df_valid['component_information']), 
         'r-', linewidth=2, alpha=0.8)

# Calculate correlation
comp_corr, comp_p = stats.pearsonr(df_valid['component_information'], df_valid['information_capacity'])

ax7.set_xlabel('Component Information', fontsize=12, fontweight='bold')
ax7.set_ylabel('Actual Information Capacity', fontsize=12, fontweight='bold')
ax7.set_title(f'Component Information vs Actual Capacity\nr = {comp_corr:.3f}, p = {comp_p:.3f}', 
              fontsize=14, fontweight='bold')
ax7.grid(True, alpha=0.3)

# 8. MODEL SUMMARY STATISTICS
ax8 = fig.add_subplot(gs[3, 2:])
ax8.axis('off')

# Create comprehensive summary
summary_text = f"""
MODEL PERFORMANCE SUMMARY
═══════════════════════════

📊 Overall Performance:
   • R² Score: {r2:.3f}
   • RMSE: {rmse:.3f}
   • MAE: {mae:.3f}
   • Sample Size: {len(df_valid)} assemblies

🎯 Prediction Quality:
   • Correlation (Actual vs Predicted): {corr_coef:.3f}
   • IDH-wt R²: {r2_wt:.3f}
   • IDH-mut R²: {r2_mut:.3f}

🔍 Feature Analysis:
   • Excitatory Ratio: {std_coefs[0]:.3f}
   • Inhibitory Ratio: {std_coefs[1]:.3f}
   • Axonal Ratio: {std_coefs[2]:.3f}
   • Component Information: {std_coefs[3]:.3f}
   • IDH Status: {std_coefs[4]:.3f}

📈 Key Insights:
   • Component Information correlation: {comp_corr:.3f}
   • Model explains {r2*100:.1f}% of variance
   • {'Good' if r2 > 0.5 else 'Moderate' if r2 > 0.3 else 'Poor'} model fit
   • {'Balanced' if abs(r2_wt - r2_mut) < 0.1 else 'Biased'} performance across IDH groups
"""

ax8.text(0.05, 0.95, summary_text, transform=ax8.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))

plt.savefig("enhanced_model_visualization.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Additional detailed analysis
print(f"\n=== DETAILED MODEL ANALYSIS ===")
print(f"Model Equation:")
print(f"Predicted Capacity = {lr.intercept_:.3f}")
for i, feature in enumerate(features):
    print(f"  + {lr.coef_[i]:.3f} × {feature}")

print(f"\nPrediction Accuracy by IDH Status:")
print(f"  IDH-wt: R² = {r2_wt:.3f}, MAE = {mae_by_idh[0]:.3f}")
print(f"  IDH-mut: R² = {r2_mut:.3f}, MAE = {mae_by_idh[1]:.3f}")

print(f"\nFeature Contributions:")
for i, feature in enumerate(features):
    print(f"  {feature}: {std_coefs[i]:.3f} (std. coef.)")

print(f"\nModel Validation:")
print(f"  Overall R²: {r2:.3f}")
print(f"  Actual vs Predicted correlation: {corr_coef:.3f}")
print(f"  Root Mean Square Error: {rmse:.3f}")
print(f"  Mean Absolute Error: {mae:.3f}")

In [ ]:
# SAVE MODEL AND SCALER FOR USE ON INDEPENDENT DATASET
# This saves all components needed to apply the trained model to new data

import joblib
import json
from datetime import datetime
import os

# Create a dictionary with all model components
model_package = {
    'model': lr,  # The trained LinearRegression model
    'scaler': scaler,  # The fitted StandardScaler
    'feature_names': features,  # Feature names in correct order
    'model_metadata': {
        'r2_score': float(r2),
        'rmse': float(rmse),
        'mae': float(mae),
        'n_samples': int(len(df_valid)),
        'feature_count': len(features),
        'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'dataset_filter': 'Grade 4 neurons only',
        'target_variable': 'information_capacity'
    },
    'model_coefficients': {
        feature: float(coef) for feature, coef in zip(features, lr.coef_)
    },
    'model_intercept': float(lr.intercept_),
    'standardized_coefficients': {
        feature: float(coef) for feature, coef in zip(features, std_coefs)
    }
}

# Optionally set the save path (change this variable as needed)
save_dir = "/userdata/gumbach/git_repos/tumor_np/revision"  # Change this path to your desired save location
model_filename = os.path.join(save_dir, 'information_capacity_model_v2.pkl')
metadata_filename = os.path.join(save_dir, 'information_capacity_model_metadata_v2.json')

joblib.dump(model_package, model_filename)

# Also save a human-readable JSON file with metadata
with open(metadata_filename, 'w') as f:
    json.dump({
        'feature_names': features,
        'model_metadata': model_package['model_metadata'],
        'model_coefficients': model_package['model_coefficients'],
        'model_intercept': model_package['model_intercept'],
        'standardized_coefficients': model_package['standardized_coefficients'],
        'usage_instructions': {
            'step1': 'Load the model: model_package = joblib.load("information_capacity_model_grade4.pkl")',
            'step2': 'Prepare your data as a DataFrame with columns matching feature_names',
            'step3': 'Extract features in correct order: X_new = df_new[model_package["feature_names"]].values',
            'step4': 'Scale features: X_new_scaled = model_package["scaler"].transform(X_new)',
            'step5': 'Make predictions: predictions = model_package["model"].predict(X_new_scaled)',
            'note': 'Ensure your independent dataset has all required features and follows same preprocessing'
        }
    }, f, indent=2)

print("=" * 80)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 80)
print(f"\nSaved files:")
print(f"  1. information_capacity_model_grade4_v2.pkl (full model package)")
print(f"  2. information_capacity_model_grade4_metadata_v2.json (metadata)")
print(f"\nModel performance on training data:")
print(f"  R² = {r2:.3f}")
print(f"  RMSE = {rmse:.3f}")
print(f"  MAE = {mae:.3f}")
print(f"  Sample size = {len(df_valid)}")
print(f"\nFeatures used (in order):")
for i, feature in enumerate(features, 1):
    print(f"  {i}. {feature}")
print(f"\nModel equation:")
print(f"  information_capacity = {lr.intercept_:.3f}", end="")
for feature, coef in zip(features, lr.coef_):
    sign = "+" if coef >= 0 else ""
    print(f" {sign} {coef:.3f} × {feature}", end="")
print("\n\n" + "=" * 80)
print("TO USE ON INDEPENDENT DATASET:")
print("=" * 80)
print("""
# Load the model
import joblib
model_package = joblib.load('information_capacity_model_grade4_v2.pkl')

# Prepare your independent dataset
# Must have these columns: excitatory_ratio, inhibitory_ratio, axonal_ratio, 
#                          component_information, idh_mutated
X_new = df_new[model_package['feature_names']].values

# Scale using the same scaler (CRITICAL!)
X_new_scaled = model_package['scaler'].transform(X_new)

# Make predictions
predictions = model_package['model'].predict(X_new_scaled)

# Compare with actual values (if available)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
if 'information_capacity' in df_new.columns:
    y_new = df_new['information_capacity'].values
    r2_new = r2_score(y_new, predictions)
    rmse_new = np.sqrt(mean_squared_error(y_new, predictions))
    mae_new = mean_absolute_error(y_new, predictions)
    print(f"Performance on independent dataset:")
    print(f"  R² = {r2_new:.3f}")
    print(f"  RMSE = {rmse_new:.3f}")
    print(f"  MAE = {mae_new:.3f}")
""")
print("=" * 80)